# Phone Studio — generation and edit worker

This is the image-worker notebook. Run one copy on each permitted worker runtime; the separate router notebook coordinates the workers and supplies the phone's single connection. This worker keeps models and results in Google Drive while running ComfyUI from runtime-local disk.

## Before running

1. In Colab choose **Runtime → Change runtime type → T4 GPU** (or another GPU available to your account). Colab does not have a supported API for an app to reserve a GPU automatically.
2. Add `NGROK_AUTHTOKEN` in **Colab Secrets**. An ngrok account/token is required for the phone to reach the temporary Colab machine.
   Optionally also add `NGROK_DOMAIN` (a free dev domain from ngrok dashboard → Gateway → Domains) so the Server URL stays the same every time this notebook runs.
3. Optionally add `CIVITAI_API_KEY` in Colab Secrets. It remains in Colab and lets the app ask the server to download Civitai model versions directly to Drive. It is also used if the default checkpoint has to be downloaded (see below), since Civitai can require a key for downloads.
4. Run every cell. The final cell first warms up your existing **WAI NSFW Illustrious v170** checkpoint with a 768 x 1024, two-step preview using the phone default sampler and scheduler. Wait for **Warm-up complete**, then save the connection values privately in the router's worker configuration. The phone connects only to the router for new jobs.

Keep this notebook running while using the phone app. Do not share the ngrok URL, tunnel password, or Phone Studio token.

## Faster T4 generation

- Selected checkpoints, LoRAs and other weights loaded through ComfyUI's standard loader are copied to `/content/phone_studio_model_cache` on first use. Later loads read the local copy. Drive remains the permanent model store; downloads from the app still go to Drive. The first copy takes time, and Colab resets clear the cache.
- The cache is capped at **24 GiB**, leaves **8 GiB** free, and falls back to Drive if it cannot cache a model. It never deletes Drive models or evicts files that a running model might still use. Adjust the limits in the GPU settings cell.
- FP16 diffusion/text encoding and cuDNN autotuning are enabled. `CHANNELS_LAST` is enabled for WAI Illustrious/SDXL; turn it off if switching to an unfamiliar architecture. Image dimensions, sampling steps, prompts and LoRA strengths are unchanged.
- Automatic `torch.compile` is requested by default on supported newer GPUs, but skipped on T4/SM7.5. Set Colab Secret `PHONE_STUDIO_AUTO_COMPILE=0` to disable it. Compilation can make startup/first use substantially slower.
- `COMPRESS_JSON_RESPONSES = True` negotiates gzip for JSON/text through ComfyUI's existing middleware, reducing metadata transfer at a small CPU cost. PNG bytes are untouched; unsupported versions keep their normal response path.
- Render preflight may reuse the model catalog for five minutes; Connect/Models **Refresh** always rescans Drive. Civitai downloads invalidate the catalog. Set `MODEL_CATALOG_CACHE_SECONDS = 0` to disable this optimization.
- Set `PHONE_STUDIO_TIMING = True` in the GPU settings cell, then rerun the bridge and launch cells to collect `[phone-studio-timing]` stage logs. GPU synchronization adds profiling overhead; leave it off for normal use. See README for the phone log and comparison procedure.
- `VRAM_MODE = 'auto'` remains the default. `'legacy'` compares the older smart-memory path without forcing residency; `'high'` requires your complete workflow to fit. Neither is automatically faster.
- `LOCAL_OUTPUT_CACHE = True` stages ordinary app PNGs locally, finishes an identical-byte copy to Drive, then serves recent images locally. The cache is bounded by `OUTPUT_CACHE_MAX_MB` and the disk reserve; misses use Drive.
- `FAST_T4_FLASH_ATTENTION = True` requests by default a pinned, patched T4/SDXL head-64 kernel. Build and tiny runtime checks are automatic; setup failure keeps SDPA. It preserves installed PyTorch/CUDA and uses bounded local build resources. Kernel rounding can change image details.
- `FAST_PNG_COMPRESSION = True` uses lossless PNG level 1 instead of stock level 4 for app saves. Pixel conversion stays the same; larger files can offset encoding gains. Enabled by default, independent of local output caching; the app switch can disable it per image.
- `FAST_FP16_INTERMEDIATES` and `FAST_FP16_VAE` are requested by default when supported. The first changes intermediate/noise/RGB rounding; the second may shorten decoding but can produce black/non-finite SDXL outputs. They need no new package and should be compared independently.
- Experimental Fast options are requested by default: `FAST_DENOISE_CACHE` approximates denoising; `FAST_FP16_ACCUMULATION` changes arithmetic. Compare quality before using them. See `SPEED_MODES.md` and `GPU_ACCELERATION_RESEARCH.md` in the app project.

To compare speed, use the same checkpoint, LoRAs, dimensions, sampler and step count. Record the first image separately (model copy/load and autotuning), then compare the **Prompt executed in ... seconds** logs for three more images with different seeds so ComfyUI actually resamples. No T4 timing has been measured for these changes yet.

Implementation references: [ComfyUI options](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/cli_args.py), [ComfyUI model loading](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/utils.py), [sample-time compilation helper](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy_api/torch_helpers/torch_compile.py), [Triton GPU support](https://github.com/triton-lang/triton#compatibility).

## Default checkpoint and startup warm-up

The default is **`waiIllustriousSDXL_v170.safetensors`** under Drive's `PhoneStudio/models/checkpoints`. If it is already there, nothing is downloaded. If it is missing, the GPU settings cell downloads it once from `DEFAULT_CHECKPOINT_URL` (a Civitai link) straight into that Drive folder, so later sessions find it and skip the download. The download can take several minutes and prints progress; if Civitai refuses it, add `CIVITAI_API_KEY` in Colab Secrets and rerun. Set `DEFAULT_CHECKPOINT_URL = ''` to turn the automatic download off, in which case a missing checkpoint stops with an error. If you move the file into a subfolder, include that prefix in `DEFAULT_CHECKPOINT`, otherwise it counts as missing and is downloaded again. Clearing `DEFAULT_CHECKPOINT` enables automatic WAI v170 name matching, with the same download if nothing matches.

The bridge lists this checkpoint first, exposes `defaultCheckpoint`, and fills missing checkpoint choices. A checkpoint explicitly selected or remembered by the phone app still takes precedence. The startup generation uses the resolved default checkpoint with no LoRAs, and completes checkpoint loading, both text encodings, sampling and VAE decoding before printing the connection details. Its temporary preview stays on Colab's local disk and does not replace the phone's latest job.

After successful startup, the bridge exposes optional instance-bound `completedWarmup` metadata. Updated phones skip a duplicate connection preview only for matching model/shape/sampler/scheduler, no LoRA and ordinary CFG guidance; old versions keep their existing behavior. Reuse stops after another workflow is submitted. This records work completed, not guaranteed permanent GPU residency.

Warm-up shifts the initial loading cost to notebook startup. ComfyUI retains cached weights subject to available RAM/VRAM; changing models, adding LoRAs, larger images or memory pressure can still trigger loading. Set the warm-up dimensions to your usual image size for more relevant cuDNN tuning.

## Two phone presets

The phone offers **Balanced (24 steps, default)** and **Fast (18 steps)** using the same standard workflow, CFG 6.5 and `dpmpp_2m`/`karras`. Fast changes the sampling result; no model or canvas is substituted. Advanced backend experiments above are requested by default and do not add phone workflows. The retired acceleration-weight and unused engine installers have been removed; existing Drive files are preserved.

## Experimental defaults and phone switch

All implemented speed experiments are now requested by default, as instructed. Diagnostics remain off because GPU synchronization slows normal use; automatic VRAM management stays selected because high/legacy modes are mutually exclusive and full residency may not fit. Unsupported hardware/builds/features print a skip reason. Conditioning caching yields to EasyCache/T4 attention, rather than stacking unsupported hooks.

The updated phone's saved **Extra acceleration** switch controls EasyCache, the T4 attention hook, fast lossless PNG and eligible conditioning caching for each newly submitted image. Missing request fields default to on for older phones. Turning it off does **not** reset launch-time FP16 accumulation/intermediates/VAE, supported-GPU compilation or the newer-GPU attention selection; turn those notebook settings off and restart for an original-quality baseline. Both phone presets keep their 24/18-step recipes. These experimental defaults can change pixels or produce invalid output; no speed or quality measurement was performed here.

## Editing through the router

Stock image-to-image, masked refinement/replacement and outpainting retain your selected WAI/SDXL checkpoint. The router uploads one validated source/mask, crops when useful, and restores unselected normalized-source pixels after generation. Edited sources and generated results are separate assets. This does not establish frontier-chatbot editing parity.

`ENABLE_FOOOCUS_INPAINT = True` requests pinned free Fooocus auxiliary weights (about 1.23 GiB plus a small head) for stronger regular-SDXL replacement/outpainting; verified setup and registered nodes determine the capability reported in health. A failed setup keeps stock edits and a visible explanation. Auxiliary weights modify the effective checkpoint; distilled checkpoints are unsupported. No Torch/CUDA replacement is performed.

`FOOOCUS_DRIVE_RESERVE_GB = 0.5` leaves 512 MiB free in Drive after each auxiliary download. It is separate from `MODEL_CACHE_RESERVE_GB = 8`, which protects runtime-local cache/build space. The Colab local-disk meter does not show Drive quota; a full Drive keeps stock edits with a visible setup message.

Edit graphs automatically bypass the per-image EasyCache, private T4 attention and conditioning-cache experiments, even when the request omits the switch. Compatibility mode disables native attention installers, automatic compilation, approximate caches and experimental FP16 options on every GPU. The saved phone acceleration switch cannot override it. Model/LoRA names, binary uploads, `/prompt`, `/history` and `/view` remain compatible. New worker-health `capabilities` fields are additive. Keep all worker passwords/tokens in router configuration; they are never sent to the phone.

## Repaired worker baseline

Upload this final cleared notebook and run its cells in order on the failed worker first. `COMPATIBILITY_MODE=True` is the ordinary PyTorch baseline on every GPU; optional installers and automatic compilation are skipped. Requested/effective options print separately. Balanced/Fast sampling, the WAI Illustrious SDXL checkpoint, seeds and negatives are unchanged.

ComfyUI is fixed to `58b176f0c8427b1e3b1e0174ded3a7794641388b`. Its required stock nodes and bridge interfaces were checked in source; live GPU compatibility still requires successful decoded warm-up. There is no moving pull. An existing different/modified checkout is preserved with a conflict; capture the healthy workers' versions and use a clean runtime-local directory for this notebook. Dependencies are resolved first with exact constraints for the existing CUDA Torch family and wheel-only installation. A conflict stops this worker instead of replacing Torch or compiling dependencies.

Private finite-stage logs live at `/content/phone_studio_worker_diagnostics`. Authenticated health includes separate runtime/instance IDs, build and actual bridge fingerprints, source revision, effective acceleration and separate host/GPU memory snapshots. These are boundary observations, not peak-use guarantees. Compare local health with tunnel/router health before changing configurations. Keep healthy workers and accepted/uncertain jobs untouched; retry only a confirmed failed worker stage.

Drive credentials and unknown files are preserved. First credential/checkpoint preparation uses an exclusive lock with a finite wait, and temporary files use instance-independent random names. The generated bridge lives on runtime-local disk; persistent user nodes are linked individually, with unknown bridge edits reported instead of erased. An occupied shared preparation lock requires the preparing instance to finish; stale locks are never removed automatically.

**Easier connection:** set the friendly `WORKER_ID` and `WORKER_NAME` in the first code cell (worker-1 / Worker 1, worker-2 / Worker 2, worker-3 / Worker 3). For existing deployments, retain the exact workerId from your router JSON. Optional `PHONE_STUDIO_WORKER_ID` and `PHONE_STUDIO_WORKER_NAME` Secrets override these labels. Keep the same ID on a restart; it identifies journal records and is not a password. After warm-up, reveal the private worker code and copy it into the router’s `PHONE_STUDIO_WORKER_CODES` Secret, one worker per line. Worker credentials stay on the router. The phone scans only the router QR. When one worker rejoins, replace only its code line and rerun idle router configuration/serving; keep healthy workers and model/smoke cells running. The existing JSON/manual configuration remains supported. Clear notebook outputs before sharing.


In [ ]:
# Mount Drive and establish persistent folders and private credentials.
import json
import os
import secrets
from pathlib import Path

from google.colab import drive, userdata

# --- Bounded worker runtime helper ---
WORKER_RUNTIME_SETUP_CODE = r'''
"""Bounded worker bootstrap. Importing this module performs no setup or installs.

The fixed ComfyUI source was inspected for Phone Studio's stock node/bridge
interfaces. GPU/runtime compatibility is pending a real warm-up; this is not a
claim that this revision is known-good on the user's workers.
"""
from __future__ import annotations

from contextlib import contextmanager
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import time
import uuid

COMFY_REPOSITORY = "https://github.com/Comfy-Org/ComfyUI.git"
COMFY_REVISION = "58b176f0c8427b1e3b1e0174ded3a7794641388b"
COMFY_REQUIREMENTS_SHA256 = "6c35de207c68baa2254fb10e25a6a48c2142b6019a6106d2547fc293393340f3"
CHECKOUT_MARKER = ".phone-studio-checkout.json"
EXPERIMENTS = ("FAST_T4_FLASH_ATTENTION", "FAST_SAGE_ATTENTION", "FAST_DENOISE_CACHE",
               "FAST_CONDITIONING_CACHE", "FAST_FP16_ACCUMULATION",
               "FAST_FP16_INTERMEDIATES", "FAST_FP16_VAE", "AUTO_COMPILE")


def effective_options(requested, compatibility_mode=True, capability=(7, 5)):
    """One authoritative requested/effective policy, independent of GPU family."""
    effective = {name: bool(requested.get(name, False)) for name in EXPERIMENTS}
    reasons = {}
    for name, value in effective.items():
        if compatibility_mode:
            effective[name] = False
            if value:
                reasons[name] = "disabled by notebook compatibility mode"
    if tuple(capability) != (7, 5) and effective["FAST_T4_FLASH_ATTENTION"]:
        effective["FAST_T4_FLASH_ATTENTION"] = False
        reasons["FAST_T4_FLASH_ATTENTION"] = "requires SM7.5"
    if capability[0] < 8:
        for name in ("FAST_SAGE_ATTENTION", "AUTO_COMPILE"):
            if effective[name]:
                effective[name] = False
                reasons[name] = "requires compute capability 8.0+"
    return {"compatibilityMode": bool(compatibility_mode), "requested": dict(requested),
            "effective": effective, "reasons": reasons}


def confirm_runtime_health(health, expected_instance, expected_runtime):
    """An open socket alone cannot prove this attempt owns the worker service."""
    if (not isinstance(health, dict) or health.get('ok') is not True
            or health.get('instanceId') != expected_instance
            or not isinstance(health.get('workerRuntime'), dict)
            or health['workerRuntime'].get('runtimeId') != expected_runtime):
        raise RuntimeError('Local worker health does not identify this launched instance; '
                           'preserve the other process and inspect the port/ownership conflict.')
    return health


def atomic_text(path, value):
    """Unique staging also works across instances with coincident process IDs."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    staging = path.with_name(path.name + "." + uuid.uuid4().hex + ".part")
    try:
        staging.write_text(value, encoding="utf-8")
        staging.replace(path)
    finally:
        staging.unlink(missing_ok=True)


@contextmanager
def exclusive_artifact(path, timeout=30):
    """Do not break another instance's lock or delete an unknown stale owner."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    owner = uuid.uuid4().hex
    deadline = time.monotonic() + timeout
    while True:
        try:
            with path.open("x", encoding="utf-8") as out:
                out.write(owner)
            break
        except FileExistsError:
            if time.monotonic() >= deadline:
                raise RuntimeError("Another instance owns artifact preparation. Preserve its lock; "
                                   "finish that preparation before retrying only this worker.") from None
            time.sleep(0.25)
    try:
        yield
    finally:
        if path.read_text(encoding="utf-8") == owner:
            path.unlink()


def _credential_values(token, password):
    # Kept self-contained for the standalone embedded bootstrap. These are the
    # same existing Latin1 Basic/header rules as the pairing helper and phone.
    if (not isinstance(token, str) or not token.strip() or len(token) > 512
            or token != token.strip() or any(not 32 <= ord(char) <= 126 for char in token)):
        raise RuntimeError("Private worker token is invalid for the authenticated header; existing credentials were preserved.")
    if (not isinstance(password, str) or not password.strip() or len(password) > 512
            or any(ord(char) < 32 or 127 <= ord(char) <= 159 or ord(char) > 255 for char in password)):
        raise RuntimeError("Private worker password contains characters unsupported by existing Basic authentication; existing credentials were preserved.")
    return token, password


def load_credentials(path, token_secret="", password_secret=""):
    import secrets
    if not isinstance(token_secret, str) or not isinstance(password_secret, str):
        raise RuntimeError("Provided private worker credentials are invalid; existing credentials were preserved.")
    path = Path(path)
    with exclusive_artifact(path.with_name(path.name + ".lock")):
        stored_missing = False
        try:
            stored = json.loads(path.read_text(encoding="utf-8"))
        except FileNotFoundError:
            stored = None
            stored_missing = True
        except (ValueError, OSError):
            raise RuntimeError("Existing private credential file cannot be read; preserve it "
                               "and inspect it privately before rerunning.") from None
        if not stored_missing:
            if not isinstance(stored, dict):
                raise RuntimeError("Existing private credential file is invalid; it was preserved.")
            _credential_values(stored.get("phone_studio_token"), stored.get("tunnel_password"))
        token, password = _credential_values(
            token_secret or (stored or {}).get("phone_studio_token") or secrets.token_urlsafe(32),
            password_secret or (stored or {}).get("tunnel_password") or secrets.token_urlsafe(24))
        # A worker's Secret overrides affect that worker only. Never rewrite a
        # shared existing credential file with a different instance's values.
        if not path.exists():
            atomic_text(path, json.dumps({"phone_studio_token": token,
                                        "tunnel_password": password}, indent=2))
        return token, password


def _redact(text):
    text = re.sub(r"https?://\S+", "[url]", text)
    text = re.sub(r"(?i)(token|password|authorization|api[_-]?key)(\s*[:=]\s*)\S+",
                  r"\1\2[redacted]", text)
    return text[-16000:]


def run_stage(command, *, stage="runtime_setup", log_root=None, timeout=180, **kwargs):
    """Finite command deadlines; retain bounded diagnostics without command/env dumps."""
    log_root = Path(log_root or "/content/phone_studio_worker_diagnostics")
    log_root.mkdir(parents=True, exist_ok=True)
    log = log_root / (stage + "." + uuid.uuid4().hex + ".log")
    print("Worker stage:", stage, "| deadline", timeout, "seconds")
    started = time.monotonic()
    try:
        result = subprocess.run(command, timeout=timeout, capture_output=True, text=True,
                                check=False, **kwargs)
    except subprocess.TimeoutExpired as error:
        atomic_text(log, json.dumps({"stage": stage, "category": "deadline", "timeout": timeout})
                    + "\n" + _redact(str(error.stdout or ""))
                    + "\n" + _redact(str(error.stderr or "")))
        raise RuntimeError(f"Worker {stage} deadline exceeded; see {log}") from None
    atomic_text(log, json.dumps({"stage": stage, "returncode": result.returncode,
                                "elapsedSeconds": round(time.monotonic() - started, 2)})
                + "\n" + _redact(result.stdout or "") + "\n" + _redact(result.stderr or ""))
    if result.returncode:
        raise RuntimeError(f"Worker {stage} failed (exit {result.returncode}); see {log}")
    return result.stdout.strip()


def _protected_package(name):
    name = name.lower().replace("_", "-")
    return name in {"torch", "torchvision", "torchaudio", "triton"} or name.startswith("nvidia-")


def torch_stack(log_root=None):
    code = """import importlib.metadata as m, json, torch
p={d.metadata['Name'].lower().replace('_','-'):d.version for d in m.distributions()}
protected={n:v for n,v in p.items() if n in {'torch','torchvision','torchaudio','triton'} or n.startswith('nvidia-')}
print(json.dumps({'packages':protected,'torchCuda':torch.version.cuda,'cudaAvailable':torch.cuda.is_available()}))
"""
    snapshot = json.loads(run_stage([sys.executable, "-c", code], stage="torch_identity",
                                   log_root=log_root, timeout=45))
    if not {"torch", "torchvision"} <= set(snapshot["packages"]) or not snapshot["torchCuda"]:
        raise RuntimeError("Worker requires an existing matched CUDA Torch/torchvision stack; "
                           "it will not install or replace that stack automatically.")
    return snapshot


def validate_dependency_report(report, stack):
    for item in report.get("install", []):
        name = str(item.get("metadata", {}).get("name", ""))
        if _protected_package(name):
            raise RuntimeError("Dependency preflight would replace/install protected GPU package "
                               + name + "; preserve the current Torch/CUDA stack.")


def prepare_checkout(root, *, log_root=None):
    root = Path(root)
    root.parent.mkdir(parents=True, exist_ok=True)
    marker = root / CHECKOUT_MARKER
    if root.is_symlink():
        raise RuntimeError("ComfyUI checkout is a symlink; preserve it and use a clean local path.")
    if root.exists():
        if not (root / ".git").is_dir():
            raise RuntimeError("Unknown existing ComfyUI directory preserved; use a clean runtime-local path.")
        revision = run_stage(["git", "-C", str(root), "rev-parse", "HEAD"],
                             stage="checkout_identity", log_root=log_root, timeout=30)
        remote = run_stage(["git", "-C", str(root), "remote", "get-url", "origin"],
                           stage="checkout_origin", log_root=log_root, timeout=30)
        changes = run_stage(["git", "-C", str(root), "diff", "--name-only", "HEAD"],
                            stage="checkout_changes", log_root=log_root, timeout=30).splitlines()
        unknown = run_stage(["git", "-C", str(root), "ls-files", "--others", "--exclude-standard"],
                            stage="checkout_untracked", log_root=log_root, timeout=30).splitlines()
        owned = {}
        if marker.is_file():
            try:
                owned = json.loads(marker.read_text(encoding="utf-8"))
            except ValueError:
                pass
        linked = owned.get("links", {}) if owned.get("revision") == COMFY_REVISION else {}
        allowed = {name for name, target in linked.items()
                   if (root / name).is_symlink() and (root / name).resolve() == Path(target).resolve()}
        bad = [p for p in changes if p.split("/", 1)[0] not in allowed]
        unknown = [p for p in unknown if p != CHECKOUT_MARKER and p not in allowed
                   and not p.startswith(".phone-studio-preserved/")]
        if revision != COMFY_REVISION or remote.rstrip("/").removesuffix(".git") != COMFY_REPOSITORY.removesuffix(".git") or bad or unknown:
            raise RuntimeError("Existing ComfyUI revision/source conflicts with this fixed notebook. "
                               "Checkout preserved; capture its package versions and use a clean "
                               "runtime-local path for this notebook. No pull/reset was performed.")
    else:
        staging = root.with_name(root.name + ".phone-studio-" + uuid.uuid4().hex)
        staging.mkdir()
        try:
            for stage, command in (
                ("checkout_init", ["git", "init", str(staging)]),
                ("checkout_remote", ["git", "-C", str(staging), "remote", "add", "origin", COMFY_REPOSITORY]),
                ("checkout_fetch", ["git", "-C", str(staging), "fetch", "--depth", "1", "origin", COMFY_REVISION]),
                ("checkout_pin", ["git", "-C", str(staging), "checkout", "--detach", COMFY_REVISION]),
            ):
                run_stage(command, stage=stage, log_root=log_root, timeout=180)
            atomic_text(staging / CHECKOUT_MARKER, json.dumps({"revision": COMFY_REVISION,
                                                            "repository": COMFY_REPOSITORY, "links": {}}))
            staging.rename(root)
        finally:
            if staging.exists():
                # This invocation created this exact sibling, never an unknown checkout.
                shutil.rmtree(staging)
    if hashlib.sha256((root / "requirements.txt").read_bytes()).hexdigest() != COMFY_REQUIREMENTS_SHA256:
        raise RuntimeError("Pinned ComfyUI requirements checksum differs; checkout was preserved.")
    if not marker.exists():
        atomic_text(marker, json.dumps({"revision": COMFY_REVISION,
                                      "repository": COMFY_REPOSITORY, "links": {}}))
    return COMFY_REVISION


def link_persistent(root, name, target, *, log_root=None):
    """Preserve shipped stock folders; never erase unknown runtime assets."""
    root, target = Path(root), Path(target)
    link = root / name
    target.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() != target.resolve():
            raise RuntimeError(f"Existing {name} link points elsewhere; it was preserved.")
    elif link.exists():
        tracked = set(run_stage(["git", "-C", str(root), "ls-files", "--", name],
                                stage="stock_" + name, log_root=log_root, timeout=30).splitlines())
        files = {p.relative_to(root).as_posix() for p in link.rglob("*") if p.is_file() or p.is_symlink()}
        if not link.is_dir() or files - tracked:
            raise RuntimeError(f"Existing {name} contains unknown files; preserved. "
                               "Move needed assets deliberately or use a clean runtime-local checkout.")
        saved = root / ".phone-studio-preserved"
        saved.mkdir(exist_ok=True)
        link.rename(saved / (name + "." + uuid.uuid4().hex))
        link.symlink_to(target, target_is_directory=True)
    else:
        link.symlink_to(target, target_is_directory=True)
    marker = root / CHECKOUT_MARKER
    record = json.loads(marker.read_text(encoding="utf-8"))
    record.setdefault("links", {})[name] = str(target.resolve())
    atomic_text(marker, json.dumps(record, indent=2))


def install_dependencies(root, *, log_root=None):
    root = Path(root)
    log_root = Path(log_root or "/content/phone_studio_worker_diagnostics")
    log_root.mkdir(parents=True, exist_ok=True)
    before = torch_stack(log_root)
    constraints = log_root / "existing-gpu-stack.txt"
    atomic_text(constraints, "\n".join(f"{n}=={v}" for n, v in sorted(before["packages"].items())) + "\n")
    report_path = log_root / "dependency-preflight.json"
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input",
               "--only-binary=:all:", "-c", str(constraints), "-r", str(root / "requirements.txt"),
               "pyngrok", "requests"]
    run_stage(command + ["--dry-run", "--report", str(report_path)],
              stage="dependency_preflight", log_root=log_root, timeout=300)
    validate_dependency_report(json.loads(report_path.read_text(encoding="utf-8")), before)
    run_stage(command, stage="dependency_install", log_root=log_root, timeout=600)
    if torch_stack(log_root) != before:
        raise RuntimeError("GPU stack identity changed unexpectedly; stop this worker and inspect private logs.")
    # Check the required startup imports, avoiding unrelated pre-existing Colab
    # package conflicts from a global pip-check policy.
    run_stage([sys.executable, "-c", "import torch, torchvision, aiohttp, numpy, requests, pyngrok"],
              stage="dependency_imports", log_root=log_root, timeout=45)
    return before


def local_custom_nodes(persistent_root, local_root, known_bridge_hashes=()):
    """Keep user nodes, but publish Phone Studio's bridge on instance-local disk."""
    persistent_root, local_root = Path(persistent_root), Path(local_root)
    local_root.mkdir(parents=True, exist_ok=True)
    inventory = []
    for item in sorted(persistent_root.iterdir()):
        if item.name == "phone_studio_bridge":
            source = item / "__init__.py"
            if source.exists() and hashlib.sha256(source.read_bytes()).hexdigest() not in known_bridge_hashes:
                raise RuntimeError("Persistent Phone Studio bridge has unknown/manual changes; "
                                   "preserved. Review those changes before replacing this worker bridge.")
            continue
        inventory.append(item.name)
        link = local_root / item.name
        if link.is_symlink() and link.resolve() == item.resolve():
            continue
        if link.exists() or link.is_symlink():
            raise RuntimeError("Local custom-node path conflicts; preserve it before retrying this worker.")
        link.symlink_to(item, target_is_directory=item.is_dir())
    return inventory
'''
_worker_setup = {'__name__': 'phone_studio_worker_runtime_setup'}
exec(compile(WORKER_RUNTIME_SETUP_CODE, '<phone-studio-worker-runtime>', 'exec'), _worker_setup)
# --- End bounded worker runtime helper ---

drive.mount('/content/drive', force_remount=False)

def colab_secret(name: str) -> str:
    # Authentication values are opaque. Keep their exact supplied bytes and
    # let the credential guard reject unsupported tokens before GPU setup.
    opaque = name in {'PHONE_STUDIO_TOKEN', 'TUNNEL_PASSWORD'}
    value = os.environ.get(name, '')
    if not opaque:
        value = value.strip()
    if value:
        return value
    try:
        value = userdata.get(name) or ''
        return value if opaque else value.strip()
    except Exception:
        return ''

DRIVE_ROOT = Path('/content/drive/MyDrive/PhoneStudio')
MODELS_ROOT = DRIVE_ROOT / 'models'
OUTPUT_ROOT = DRIVE_ROOT / 'output'
INPUT_ROOT = DRIVE_ROOT / 'input'
USER_ROOT = DRIVE_ROOT / 'user'
CUSTOM_NODES_ROOT = DRIVE_ROOT / 'custom_nodes'
RUNTIME_ROOT = Path('/content/ComfyUI')
for folder in (MODELS_ROOT, OUTPUT_ROOT, INPUT_ROOT, USER_ROOT, CUSTOM_NODES_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
for folder in ('checkpoints', 'loras', 'vae', 'upscale_models', 'controlnet'):
    (MODELS_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Compatibility mode always wins over saved phone acceleration switches.
COMPATIBILITY_MODE = True
import uuid
WORKER_RUNTIME_ID = globals().get('WORKER_RUNTIME_ID') or uuid.uuid4().hex
WORKER_DIAGNOSTICS_ROOT = Path('/content/phone_studio_worker_diagnostics')
RUNTIME_CUSTOM_NODES_ROOT = Path('/content/phone_studio_custom_nodes')
credentials_path = DRIVE_ROOT / '.phone_studio_credentials.json'
PHONE_STUDIO_TOKEN, TUNNEL_PASSWORD = _worker_setup['load_credentials'](
    credentials_path, colab_secret('PHONE_STUDIO_TOKEN'), colab_secret('TUNNEL_PASSWORD'))
TUNNEL_USER = 'studio'

# --- Private worker pairing helper ---
CONNECTION_PAIRING_CODE = r'''
"""Private, bounded notebook connection codes. Importing this module does no setup.

Codes contain the existing credentials. They are a local transfer format, not an
authentication bypass, public URL or remotely hosted QR service.
"""
from __future__ import annotations

import base64
from contextlib import contextmanager
import html
import ipaddress
import json
import os
from pathlib import Path
import re
import secrets
import time
from urllib.parse import unquote, urlsplit, urlunsplit
import uuid

PREFIX = "PHONE-STUDIO:1:"
MAX_CODE_LENGTH = 8192
MAX_PAYLOAD_BYTES = 4096
QR_PACKAGE = "qrcode==8.2"
_FIELDS = {"version", "kind", "name", "serverUrl", "tunnelUser", "tunnelPassword", "studioToken", "workerId"}


class PairingError(ValueError):
    """Messages deliberately never include the supplied code or credentials."""


def _object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise PairingError("Connection code contains duplicate fields.")
        result[key] = value
    return result


def _text(value, label, limit, *, trimmed=False):
    if (not isinstance(value, str) or not value.strip() or len(value) > limit
            or any(ord(char) < 32 or 127 <= ord(char) <= 159 or 0xD800 <= ord(char) <= 0xDFFF for char in value)
            or (trimmed and value != value.strip())):
        raise PairingError(f"Connection {label} is invalid.")
    return value


def _token(value):
    value = _text(value, "token", 512, trimmed=True)
    if any(not 32 <= ord(char) <= 126 for char in value):
        raise PairingError("Connection token must use visible ASCII text for the authenticated header.")
    return value


def _basic_text(value, label, limit, *, trimmed=False):
    value = _text(value, label, limit, trimmed=trimmed)
    if any(ord(char) > 255 for char in value):
        raise PairingError(f"Connection {label} contains characters unsupported by the existing Basic authentication.")
    return value


def _basic_user(value):
    value = _basic_text(value, "user", 128, trimmed=True)
    if ":" in value:
        raise PairingError("Connection user cannot contain a colon.")
    return value


def validate_basic_credentials(user, password):
    """Match the phone/server's existing Latin1 protocol without altering secrets."""
    return _basic_user(user), _basic_text(password, "password", 512)


def basic_authorization(user, password):
    user, password = validate_basic_credentials(user, password)
    encoded = base64.b64encode((user + ":" + password).encode("latin1", errors="strict")).decode("ascii")
    return "Basic " + encoded


def clean_server_url(value):
    value = _text(value, "address", 2048, trimmed=True)
    if not value.startswith("https://") or any(char.isspace() for char in value) or "\\" in value:
        raise PairingError("Use a clean HTTPS server address.")
    try:
        parsed = urlsplit(value)
        hostname, port = parsed.hostname, parsed.port
    except ValueError:
        raise PairingError("Use a clean HTTPS server address.") from None
    if (parsed.scheme != "https" or not hostname or parsed.username is not None
            or parsed.password is not None or parsed.query or parsed.fragment
            or "?" in value or "#" in value
            or not re.fullmatch(r"[A-Za-z0-9.:-]+", hostname)
            or (port is not None and not 1 <= port <= 65535)):
        raise PairingError("Use a clean HTTPS server address without credentials or query fields.")
    try:
        if ":" in hostname:
            ipaddress.IPv6Address(hostname)
        elif (len(hostname) > 253 or any(not re.fullmatch(r"[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?", label)
                                      for label in hostname.removesuffix(".").split("."))):
            raise ValueError
        if parsed.netloc.endswith(":"):
            raise ValueError
    except ValueError:
        raise PairingError("Connection address host is invalid.") from None
    try:
        decoded_path = unquote(parsed.path, encoding="utf-8", errors="strict")
    except UnicodeError:
        raise PairingError("Connection address path encoding is invalid.") from None
    if ("\\" in decoded_path or any(ord(char) < 32 or 127 <= ord(char) <= 159 for char in decoded_path)
            or any(part in (".", "..") for part in decoded_path.split("/"))
            or re.search(r"%(?![0-9A-Fa-f]{2})", parsed.path)):
        raise PairingError("Connection address path is invalid.")
    host = "[" + ipaddress.IPv6Address(hostname).compressed + "]" if ":" in hostname else hostname.lower().removesuffix(".")
    netloc = host + (":" + str(port) if port is not None and port != 443 else "")
    return urlunsplit(("https", netloc, parsed.path.rstrip("/"), "", ""))


def validate_pairing(body, *, expected_kind=None):
    if not isinstance(body, dict) or set(body) - _FIELDS:
        raise PairingError("Connection code contains unsupported fields.")
    if type(body.get("version")) is not int or body["version"] != 1:
        raise PairingError("Connection code version is unsupported.")
    kind = body.get("kind")
    if kind not in ("router", "worker"):
        raise PairingError("Connection code has an invalid role.")
    if expected_kind is not None and kind != expected_kind:
        raise PairingError("Use a router code for the phone or worker codes for the router.")
    required = _FIELDS - {"workerId"}
    if not required <= set(body) or (kind == "router" and "workerId" in body):
        raise PairingError("Connection code fields do not match its role.")
    result = {"version": 1, "kind": kind,
              "name": _text(body["name"], "name", 80, trimmed=True),
              "serverUrl": clean_server_url(body["serverUrl"]),
              "tunnelUser": _basic_user(body["tunnelUser"]),
              "tunnelPassword": _basic_text(body["tunnelPassword"], "password", 512),
              "studioToken": _token(body["studioToken"])}
    if kind == "worker":
        worker_id = body.get("workerId")
        if not isinstance(worker_id, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,127}", worker_id):
            raise PairingError("Connection worker ID is invalid.")
        result["workerId"] = worker_id
    return result


def encode_pairing(body):
    value = json.dumps(validate_pairing(body), ensure_ascii=False, separators=(",", ":"))
    data = value.encode("utf-8")
    if len(data) > MAX_PAYLOAD_BYTES:
        raise PairingError("Connection code is too large.")
    return PREFIX + base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")


def decode_pairing(code, *, expected_kind=None):
    if not isinstance(code, str) or len(code) > MAX_CODE_LENGTH:
        raise PairingError("Connection code is missing or too large.")
    code = code.strip()
    if not code.startswith(PREFIX):
        raise PairingError("Paste a Phone Studio connection code.")
    encoded = code[len(PREFIX):]
    if not encoded or not re.fullmatch(r"[A-Za-z0-9_-]+", encoded):
        raise PairingError("Connection code encoding is invalid.")
    try:
        data = base64.b64decode(encoded + "=" * (-len(encoded) % 4), altchars=b"-_", validate=True)
        if len(data) > MAX_PAYLOAD_BYTES:
            raise PairingError("Connection code is too large.")
        if base64.urlsafe_b64encode(data).decode("ascii").rstrip("=") != encoded:
            raise PairingError("Connection code encoding is invalid.")
        body = json.loads(data.decode("utf-8", errors="strict"), object_pairs_hook=_object,
                          parse_constant=lambda _: (_ for _ in ()).throw(PairingError("Connection code JSON is invalid.")))
    except (ValueError, UnicodeError, RecursionError) as error:
        if isinstance(error, PairingError):
            raise
        raise PairingError("Connection code encoding or JSON is invalid.") from None
    return validate_pairing(body, expected_kind=expected_kind)


def workers_from_codes(text):
    if not isinstance(text, str) or len(text) > MAX_CODE_LENGTH * 3 + 8:
        raise PairingError("Provide one to three private worker codes, one per line.")
    codes = [line.strip() for line in text.splitlines() if line.strip()]
    if not 1 <= len(codes) <= 3:
        raise PairingError("Provide one to three private worker codes, one per line.")
    entries = []
    ids, addresses = set(), set()
    for code in codes:
        body = decode_pairing(code, expected_kind="worker")
        if body["workerId"] in ids or body["serverUrl"] in addresses:
            raise PairingError("Worker codes must have distinct IDs and server addresses.")
        ids.add(body["workerId"])
        addresses.add(body["serverUrl"])
        entries.append({key: value for key, value in body.items() if key not in ("version", "kind")})
    return entries


@contextmanager
def _credential_lock(path, timeout):
    if not isinstance(timeout, (int, float)) or not 0 <= timeout <= 30:
        raise PairingError("Credential preparation timeout is invalid.")
    owner = uuid.uuid4().hex
    deadline = time.monotonic() + timeout
    while True:
        if path.is_symlink():
            raise PairingError("Private credential lock is a symlink; preserve it and inspect it privately.")
        try:
            descriptor = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
            with os.fdopen(descriptor, "w", encoding="ascii") as output:
                output.write(owner)
            break
        except FileExistsError:
            if time.monotonic() >= deadline:
                raise PairingError("Another router owns credential preparation; preserve its lock and finish that preparation.") from None
            time.sleep(min(0.1, max(0, deadline - time.monotonic())))
    try:
        yield
    finally:
        try:
            if not path.is_symlink() and path.read_text(encoding="ascii") == owner:
                path.unlink()
        except (OSError, UnicodeError):
            pass


def load_router_credentials(path, token_secret="", password_secret="", *, timeout=10, tunnel_user="studio"):
    """Keep existing Secrets; generate missing credentials once under persistent state.

    Per-notebook Secret overrides never overwrite an existing stored credential file.
    Unknown/corrupt files and occupied locks are preserved rather than reset.
    """
    if not isinstance(token_secret, str) or not isinstance(password_secret, str):
        raise PairingError("Provided router credentials are invalid.")
    _basic_user(tunnel_user)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with _credential_lock(path.with_name(path.name + ".lock"), timeout):
        if path.is_symlink():
            raise PairingError("Private credential file is a symlink; preserve it and inspect it privately.")
        try:
            if path.stat().st_size > MAX_PAYLOAD_BYTES:
                raise PairingError("Stored router credentials are invalid; the file was preserved.")
            stored = json.loads(path.read_text(encoding="utf-8"), object_pairs_hook=_object)
            if not isinstance(stored, dict) or set(stored) != {"version", "phone_studio_token", "tunnel_password"} or type(stored["version"]) is not int or stored["version"] != 1:
                raise PairingError("Stored router credentials are invalid; the file was preserved.")
            _token(stored["phone_studio_token"])
            _basic_text(stored["tunnel_password"], "password", 512)
        except FileNotFoundError:
            stored = None
        except (OSError, ValueError, UnicodeError):
            raise PairingError("Stored router credentials cannot be read; preserve the file and inspect it privately.") from None
        token = _token(token_secret or (stored or {}).get("phone_studio_token") or secrets.token_urlsafe(32))
        password = _basic_text(password_secret or (stored or {}).get("tunnel_password") or secrets.token_urlsafe(24), "password", 512)
        if stored is None:
            staging = path.with_name(path.name + "." + uuid.uuid4().hex + ".part")
            try:
                descriptor = os.open(staging, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
                with os.fdopen(descriptor, "w", encoding="utf-8") as output:
                    json.dump({"version": 1, "phone_studio_token": token, "tunnel_password": password}, output)
                staging.replace(path)
            finally:
                staging.unlink(missing_ok=True)
        return token, password


def pairing_html(body, *, with_qr=True):
    """Local SVG only. Missing QR package/capacity never blocks paste-code pairing."""
    value = validate_pairing(body)
    code = encode_pairing(value)
    qr_html = ""
    if with_qr:
        try:
            import qrcode
            from qrcode.image.svg import SvgPathFillImage
            qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_M, box_size=6, border=4)
            qr.add_data(code, optimize=0)
            qr.make(fit=True)
            svg = qr.make_image(image_factory=SvgPathFillImage).to_string(encoding="unicode")
            qr_html = '<div style="background:white;padding:12px;max-width:480px">' + svg + '</div>'
        except Exception:
            qr_html = '<p>QR preview is unavailable. Use the private code below.</p>'
    action = "Scan in Phone Studio → Connect" if value["kind"] == "router" else "Copy into the router’s PHONE_STUDIO_WORKER_CODES Secret"
    return ('<section style="font-family:system-ui;padding:16px;border:1px solid #b7c6d6;border-radius:16px">'
            '<h3>' + html.escape(action) + '</h3><p>' + html.escape(value["name"]) + ' · '
            + html.escape(value["serverUrl"]) + '</p><p>Private: this code contains connection credentials. '
            'Keep this notebook output private and clear outputs before sharing.</p>' + qr_html
            + '<details><summary>Reveal private connection code for copy/paste</summary>'
            '<textarea readonly rows="5" style="width:100%;box-sizing:border-box;margin-top:12px">'
            + html.escape(code) + '</textarea></details></section>')


def display_pairing(body, *, with_qr=True):
    from IPython.display import HTML, display
    display(HTML(pairing_html(body, with_qr=with_qr)))
'''
_pairing = {'__name__': 'phone_studio_connection_pairing'}
exec(compile(CONNECTION_PAIRING_CODE, '<phone-studio-connection-pairing>', 'exec'), _pairing)
# Friendly IDs are stable through restarts. Use your existing router workerId.
WORKER_ID = 'worker-1'  # Change to worker-2 / worker-3 in the other copies.
WORKER_NAME = 'Worker 1'
WORKER_ID = colab_secret('PHONE_STUDIO_WORKER_ID') or WORKER_ID
WORKER_NAME = colab_secret('PHONE_STUDIO_WORKER_NAME') or WORKER_NAME
# Validate friendly configuration before downloading or warming the worker.
_pairing['validate_pairing']({'version': 1, 'kind': 'worker', 'name': WORKER_NAME,
    'workerId': WORKER_ID, 'serverUrl': 'https://setup.invalid', 'tunnelUser': TUNNEL_USER,
    'tunnelPassword': TUNNEL_PASSWORD, 'studioToken': PHONE_STUDIO_TOKEN})
# --- End private worker pairing helper ---

NGROK_AUTHTOKEN = colab_secret('NGROK_AUTHTOKEN')
CIVITAI_API_KEY = colab_secret('CIVITAI_API_KEY')
# Explicit opt-in only; compatibility mode disables compilation on every GPU.
PHONE_STUDIO_AUTO_COMPILE = colab_secret('PHONE_STUDIO_AUTO_COMPILE') or '0'
if not NGROK_AUTHTOKEN:
    raise RuntimeError('Add NGROK_AUTHTOKEN in Colab Secrets, then rerun this cell.')

print(f'Drive workspace: {DRIVE_ROOT}')
print('Persistent models:', MODELS_ROOT)


In [ ]:
# Fixed source and a wheel-only dependency preflight preserve the current GPU stack.
import ctypes.util
import shutil
import subprocess
import sys

def run(command, **kwargs):
    return _worker_setup['run_stage'](command, stage='optional_setup',
        log_root=WORKER_DIAGNOSTICS_ROOT, timeout=300, **kwargs)

missing_system = []
if not shutil.which('git'):
    missing_system.append('git')
if not ctypes.util.find_library('GL'):
    missing_system.append('libgl1')
if not ctypes.util.find_library('glib-2.0'):
    missing_system.append('libglib2.0-0')
if missing_system:
    _worker_setup['run_stage'](['apt-get', 'update', '-y'], stage='system_metadata',
        log_root=WORKER_DIAGNOSTICS_ROOT, timeout=180)
    _worker_setup['run_stage'](['apt-get', 'install', '-y', '--no-install-recommends', *missing_system],
        stage='system_dependencies', log_root=WORKER_DIAGNOSTICS_ROOT, timeout=300)

COMFY_REVISION = _worker_setup['prepare_checkout'](RUNTIME_ROOT, log_root=WORKER_DIAGNOSTICS_ROOT)
WORKER_GPU_STACK = _worker_setup['install_dependencies'](RUNTIME_ROOT, log_root=WORKER_DIAGNOSTICS_ROOT)
for name, target in (('models', MODELS_ROOT), ('output', OUTPUT_ROOT), ('input', INPUT_ROOT),
                     ('user', USER_ROOT), ('custom_nodes', RUNTIME_CUSTOM_NODES_ROOT)):
    _worker_setup['link_persistent'](RUNTIME_ROOT, name, target, log_root=WORKER_DIAGNOSTICS_ROOT)
print('Fixed ComfyUI source:', COMFY_REVISION)
print('Torch:', WORKER_GPU_STACK['packages']['torch'], '| CUDA:', WORKER_GPU_STACK['torchCuda'])
print('Source interfaces checked; GPU compatibility still requires the decoded warm-up below.')


In [ ]:
# GPU settings for Colab. No replacement of Colab's CUDA/PyTorch installation.
import torch

ENABLE_FOOOCUS_INPAINT = True  # Free pinned regular-SDXL inpaint auxiliary; honest stock fallback.
FOOOCUS_DRIVE_RESERVE_GB = 0.5  # Drive quota reserve for optional weights; local cache/build keeps 8 GiB.
LOCAL_MODEL_CACHE = True
MODEL_CATALOG_CACHE_SECONDS = 300  # Render preflight only; explicit Refresh always scans Drive.
PHONE_STUDIO_TIMING = False       # Diagnostic GPU synchronization adds overhead; enable to profile.
COMPRESS_JSON_RESPONSES = True   # Negotiated gzip for JSON/text only; image bytes stay untouched.
LOCAL_OUTPUT_CACHE = True         # Encode locally, finish the Drive copy, then serve recent PNGs locally.
OUTPUT_CACHE_MAX_MB = 256          # CPU disk cache only; no extra model or GPU allocation.
FAST_PNG_COMPRESSION = True      # Optional lossless level 1; larger PNGs may slow transfer.
FAST_DENOISE_CACHE = False        # Experimental EasyCache: faster approximate denoising may change images.
FAST_CONDITIONING_CACHE = False  # Bounded experimental cache; EasyCache/T4 attention take precedence.
MODEL_CACHE_MAX_GB = 24
MODEL_CACHE_RESERVE_GB = 8
MODEL_CACHE_ROOT = Path('/content/phone_studio_model_cache')
VRAM_MODE = 'auto'       # 'legacy' compares old smart memory; 'high' requires a workflow that fits.
FAST_FP16_ACCUMULATION = False  # Optional Fast arithmetic: may change images or overflow.
FAST_FP16_INTERMEDIATES = False  # Optional lower-precision buffers; noise/RGB rounding changes output.
FAST_FP16_VAE = False            # Optional faster decode; some SDXL VAEs produce black/non-finite images.
FAST_T4_FLASH_ATTENTION = False  # Optional native SDXL/T4 kernel; rounding can change images.
CHANNELS_LAST = True    # WAI Illustrious is an SDXL image checkpoint.
                       # Set False if switching to unfamiliar/video architectures with 5D weights.

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then rerun the notebook.')
if VRAM_MODE not in {'auto', 'legacy', 'high'}:
    raise ValueError("VRAM_MODE must be 'auto', 'legacy' or 'high'.")
if OUTPUT_CACHE_MAX_MB <= 0:
    raise ValueError('OUTPUT_CACHE_MAX_MB must be positive.')
if MODEL_CACHE_MAX_GB <= 0 or MODEL_CACHE_RESERVE_GB < 0:
    raise ValueError('Cache capacity must be positive and reserved disk space cannot be negative.')

FAST_SAGE_ATTENTION = False  # Explicit optional installer; off for the baseline.
SAGE_ATTENTION_AVAILABLE = False
T4_FLASH_ATTENTION_AVAILABLE = False
cc_major, cc_minor = torch.cuda.get_device_capability(0)
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
# --- Requested and effective worker options ---
WORKER_OPTIONS = _worker_setup['effective_options']({
    name: globals()[name] for name in _worker_setup['EXPERIMENTS'] if name != 'AUTO_COMPILE'
} | {'AUTO_COMPILE': PHONE_STUDIO_AUTO_COMPILE == '1'}, COMPATIBILITY_MODE, (cc_major, cc_minor))
for name, enabled in WORKER_OPTIONS['effective'].items():
    if name != 'AUTO_COMPILE':
        globals()[name] = enabled
PHONE_STUDIO_AUTO_COMPILE = '1' if WORKER_OPTIONS['effective']['AUTO_COMPILE'] else '0'
print('Worker options:', json.dumps(WORKER_OPTIONS, sort_keys=True))
# --- End requested and effective worker options ---
print(f'GPU: {gpu_name} | {vram_gb:.1f} GiB VRAM | compute capability {cc_major}.{cc_minor}')
if PHONE_STUDIO_TIMING:
    print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
    print('ComfyUI revision:', subprocess.check_output(['git', '-C', str(RUNTIME_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

if FAST_SAGE_ATTENTION and cc_major >= 8:
    print('Explicit SageAttention installation requested on supported hardware.')
    try:
        run([sys.executable, '-m', 'pip', 'install', '-q', 'sageattention==2.2.0', '--no-build-isolation'])
        import sageattention
        SAGE_ATTENTION_AVAILABLE = True
        print('SageAttention2 installed; using --use-sage-attention.')
    except Exception:
        print('Optional SageAttention unavailable; private stage log has the reason. Using PyTorch attention.')
else:
    print('Ordinary PyTorch attention; optional SageAttention installer skipped.')

# --- Optional T4 Fast attention; self-contained pinned installer ---
# The source is embedded so uploading this notebook alone is sufficient.
# Nothing is built/imported/dispatched unless the explicit Fast switch is on.
if FAST_T4_FLASH_ATTENTION:
    if (cc_major, cc_minor) != (7, 5):
        print('T4 Fast attention skipped: requires SM7.5. Keeping the existing GPU attention choice.')
    else:
        T4_FLASH_INSTALLER_CODE = r'''
"""Optional, pinned T4 attention build for Colab; importing this file installs nothing.

Call install_t4_flash_attention() before starting ComfyUI. It returns a validated
manifest or None and preserves the existing PyTorch/CUDA packages. Three tiny GPU
numeric/stream checks run only when the user invokes the opt-in installer; they
are not a speed benchmark or a full-image quality check.

Source: https://github.com/ssiu/flash-attention-turing
"""

import hashlib
import importlib.util
import json
import os
from pathlib import Path
import re
import shutil
import signal
import subprocess
import sys
import tempfile
import time


T4_FLASH_COMMIT = "9ef98fcb506bb1e2fe3cece50935e2935bf6b124"
T4_FLASH_CUTLASS_COMMIT = "df18f5e4f5de76bed8be1de8e4c245f2f5ec3020"
T4_FLASH_PATCH = "phone-studio-t4-forward-v1"
T4_FLASH_MODULE = "phone_studio_flash_attn_turing"
T4_FLASH_ROOT = Path("/content/phone_studio_t4_flash")
T4_FLASH_RUNTIME = T4_FLASH_ROOT / "runtime"
T4_FLASH_STAMP = T4_FLASH_ROOT / "install.json"
_GIB = 1024 ** 3
_SOURCE_HASHES = {
    "setup.py": "3788d57ad0d1fae5313cc3247fb4f3f4f8705d51f43deba3ad5399125134fddb",
    "csrc/flash_attn/flash_api.cpp": "a00f82a9be15f56fdd2e688cc098525b9939b708873e678a64c8da8f941cc9f4",
    "csrc/flash_attn/src/flash_fwd_launch_template.h":
        "edb12023488232b63c3e8c8c72801c93c559f35da13e302fcaf28100803cd0e8",
}


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _replace_once(text, before, after, label):
    if text.count(before) != 1:
        raise RuntimeError(f"Pinned source did not match the {label} patch")
    return text.replace(before, after, 1)


def _patch_source(source_dir):
    """Check upstream bytes, then patch only the audited forward entry point."""
    for relative, expected in _SOURCE_HASHES.items():
        if _sha256(source_dir / relative) != expected:
            raise RuntimeError(f"Pinned source checksum failed: {relative}")

    setup_path = source_dir / "setup.py"
    setup = setup_path.read_text(encoding="utf-8")
    setup = _replace_once(setup, '\n    name="flash_attn_turing",',
                          f'\n    name="{T4_FLASH_MODULE}",', "distribution name")
    setup = _replace_once(setup, '\n            name="flash_attn_turing",',
                          f'\n            name="{T4_FLASH_MODULE}",', "native extension name")
    # One noncausal head64 forward specialization is the actual SDXL target.
    # Remove every other compilation unit; the matching API trimming below also
    # removes their references so the resulting library has no unresolved symbols.
    units = [
        'csrc/flash_attn/src/flash_fwd_hdim64_fp16_causal_sm75.cu',
        'csrc/flash_attn/src/flash_fwd_hdim96_fp16_sm75.cu',
        'csrc/flash_attn/src/flash_fwd_hdim96_fp16_causal_sm75.cu',
        'csrc/flash_attn/src/flash_fwd_hdim128_fp16_sm75.cu',
        'csrc/flash_attn/src/flash_fwd_hdim128_fp16_causal_sm75.cu',
        'csrc/flash_attn/src/flash_bwd_hdim64_fp16_sm75.cu',
        'csrc/flash_attn/src/flash_bwd_hdim64_fp16_causal_sm75.cu',
        'csrc/flash_attn/src/flash_bwd_hdim96_fp16_sm75.cu',
        'csrc/flash_attn/src/flash_bwd_hdim96_fp16_causal_sm75.cu',
        'csrc/flash_attn/src/flash_bwd_hdim128_fp16_sm75.cu',
    ]
    for unit in units:
        setup = _replace_once(setup, f'                      "{unit}",\n', '',
                              f'unused compilation unit {unit}')
    setup = _replace_once(setup,
        '                      "csrc/flash_attn/src/flash_bwd_hdim128_fp16_causal_sm75.cu"\n',
        '', 'last unused compilation unit')
    setup_path.write_text(setup, encoding="utf-8")

    api_path = source_dir / "csrc/flash_attn/flash_api.cpp"
    api = api_path.read_text(encoding="utf-8")
    api = _replace_once(
        api, "#include <torch/extension.h>\n",
        "#include <torch/extension.h>\n#include <c10/cuda/CUDAGuard.h>\n"
        "#include <ATen/cuda/CUDAContext.h>\n#include <cmath>\n#include <cstdint>\n#include <limits>\n",
        "native device guard includes",
    )
    checks = """    TORCH_CHECK(q.dim() == 4 && k.dim() == 4 && v.dim() == 4,
                "Phone Studio T4 attention requires rank-4 tensors");
    TORCH_CHECK(q.is_cuda() && k.is_cuda() && v.is_cuda(),
                "Phone Studio T4 attention requires CUDA tensors");
    TORCH_CHECK(q.device() == k.device() && q.device() == v.device(),
                "Phone Studio T4 attention requires one CUDA device");
    TORCH_CHECK(q.scalar_type() == torch::kFloat16 &&
                k.scalar_type() == torch::kFloat16 && v.scalar_type() == torch::kFloat16,
                "Phone Studio T4 attention requires FP16 tensors");
    TORCH_CHECK(q.is_contiguous() && k.is_contiguous() && v.is_contiguous(),
                "Phone Studio T4 attention requires contiguous B,L,H,D tensors");
    TORCH_CHECK(reinterpret_cast<std::uintptr_t>(q.data_ptr()) % 16 == 0 &&
                reinterpret_cast<std::uintptr_t>(k.data_ptr()) % 16 == 0 &&
                reinterpret_cast<std::uintptr_t>(v.data_ptr()) % 16 == 0,
                "Phone Studio T4 attention requires 16-byte-aligned Q/K/V tensors");
    TORCH_CHECK(!q.requires_grad() && !k.requires_grad() && !v.requires_grad(),
                "Phone Studio T4 attention is inference-only");
    TORCH_CHECK(!is_causal && std::isfinite(softmax_scale) && softmax_scale > 0.0f,
                "Phone Studio T4 attention supports unmasked forward attention only");
    for (const auto &tensor : {q, k, v}) {
        TORCH_CHECK(tensor.numel() < std::numeric_limits<int>::max(),
                    "Phone Studio T4 attention tensor exceeds kernel indexing limits");
        for (const auto size : tensor.sizes()) {
            TORCH_CHECK(size > 0 && size < std::numeric_limits<int>::max(),
                        "Phone Studio T4 attention requires nonempty bounded tensors");
        }
    }
    TORCH_CHECK(q.size(3) == 64,
                "Phone Studio T4 attention supports SDXL head size 64");
    const c10::cuda::CUDAGuard device_guard(q.device());
    const auto properties = at::cuda::getCurrentDeviceProperties();
    TORCH_CHECK(properties->major == 7 && properties->minor == 5,
                "Phone Studio T4 attention requires SM7.5");
"""
    api = _replace_once(api, "    auto device = q.device();\n",
                        checks + "    auto device = q.device();\n", "forward argument checks")
    api = _replace_once(
        api,
        '    m.def("bwd", &mha_bwd, "Backward pass");\n'
        '    m.def("varlen_fwd", &mha_varlen_fwd, "Varlen forward pass");\n'
        '    m.def("varlen_bwd", &mha_varlen_bwd, "Varlen backward pass");\n',
        f'    m.attr("__phone_studio_patch__") = "{T4_FLASH_PATCH}";\n'
        f'    m.attr("__phone_studio_commit__") = "{T4_FLASH_COMMIT}";\n',
        "forward-only exported API",
    )
    # Keep the audited parameter builder and regular forward function. All
    # backward/varlen/general-dispatch functions are excluded from this private
    # inference build; their original files remain in the pinned checkout.
    anchors = ['void set_params_dgrad(', 'std::vector<at::Tensor>\nmha_fwd(',
               'std::vector<at::Tensor>\nmha_bwd(',
               'PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {']
    if any(api.count(anchor) != 1 for anchor in anchors):
        raise RuntimeError('Pinned source did not match the forward-only API boundaries')
    prefix_end, forward_start, forward_end, binding_start = [api.index(a) for a in anchors]
    if not prefix_end < forward_start < forward_end < binding_start:
        raise RuntimeError('Pinned source forward-only API boundaries were reordered')
    forward = _replace_once(api[forward_start:forward_end], '    run_mha_fwd(params);',
                            '    run_mha_fwd_<64, false>(params);', 'head64 direct dispatch')
    api = api[:prefix_end] + forward + api[binding_start:]
    api_path.write_text(api, encoding="utf-8")

    launch_path = source_dir / "csrc/flash_attn/src/flash_fwd_launch_template.h"
    launch = launch_path.read_text(encoding="utf-8")
    launch = _replace_once(
        launch, "#pragma once\n",
        "#pragma once\n#include <ATen/cuda/CUDAContext.h>\n"
        "#include <c10/cuda/CUDAException.h>\n", "stream/error-check includes",
    )
    launch = _replace_once(
        launch, "void run_flash_fwd(Flash_fwd_params &params) {\n",
        "void run_flash_fwd(Flash_fwd_params &params) {\n"
        "    const cudaStream_t stream = at::cuda::getCurrentCUDAStream();\n",
        "current PyTorch CUDA stream",
    )
    launch = _replace_once(
        launch,
        "        cudaFuncSetAttribute(flash_fwd_kernel<Kernel_traits, Is_causal, Is_even_MN>, "
        "cudaFuncAttributeMaxDynamicSharedMemorySize, maxbytes);",
        "        C10_CUDA_CHECK(cudaFuncSetAttribute("
        "flash_fwd_kernel<Kernel_traits, Is_causal, Is_even_MN>, "
        "cudaFuncAttributeMaxDynamicSharedMemorySize, maxbytes));",
        "shared-memory launch setup check",
    )
    launch = _replace_once(launch, "<<<dimGrid, dimBlock, maxbytes>>>",
                           "<<<dimGrid, dimBlock, maxbytes, stream>>>", "forward CUDA launch")
    launch = _replace_once(
        launch, "                                                                                    params.is_causal);\n    });",
        "                                                                                    params.is_causal);\n"
        "        C10_CUDA_KERNEL_LAUNCH_CHECK();\n    });", "forward launch error check",
    )
    launch_path.write_text(launch, encoding="utf-8")


def _stop_process_tree(process):
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=10)


def _run(command, *, cwd=None, env=None, timeout=180, log_path=None):
    """Terminate the whole compiler tree on timeout; never leave a build running."""
    if log_path is None:
        return subprocess.run(command, cwd=cwd, env=env, check=True,
                              timeout=timeout, text=True, capture_output=True).stdout.strip()
    with Path(log_path).open("a", encoding="utf-8") as output:
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=output,
                                   stderr=subprocess.STDOUT, start_new_session=True)
        try:
            code = process.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            _stop_process_tree(process)
            raise RuntimeError(f"T4 attention build exceeded {timeout}s; see {log_path}")
        except BaseException:
            # A notebook interrupt cancels the entire compiler tree before it
            # propagates; ordinary ComfyUI must not start beside orphan nvcc jobs.
            _stop_process_tree(process)
            raise
    if code:
        raise RuntimeError(f"T4 attention command failed ({code}); see {log_path}")
    return ""


def _host_available_bytes():
    available = 0
    for line in Path("/proc/meminfo").read_text(encoding="ascii").splitlines():
        if line.startswith("MemAvailable:"):
            available = int(line.split()[1]) * 1024
            break
    # A container can have much less memory than /proc/meminfo describes.
    for limit_path, usage_path in (
        ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
        ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes"),
    ):
        try:
            limit = Path(limit_path).read_text(encoding="ascii").strip()
            usage = Path(usage_path).read_text(encoding="ascii").strip()
            if limit != "max" and int(limit) > 0:
                available = min(available, max(0, int(limit) - int(usage)))
        except (OSError, ValueError):
            pass
    return available


def _validate_native(binary, env):
    # This source is executed only in the user's opt-in Colab session. A fresh
    # interpreter avoids stale loaded extensions and isolates its RNG state.
    program = f"""
import importlib.util
import json
import sys
import torch
import torch.nn.functional as F
spec = importlib.util.spec_from_file_location({T4_FLASH_MODULE!r}, sys.argv[1])
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
assert module.__phone_studio_patch__ == {T4_FLASH_PATCH!r}
assert module.__phone_studio_commit__ == {T4_FLASH_COMMIT!r}
assert callable(module.fwd)
assert torch.cuda.get_device_capability() == (7, 5)
assert torch.cuda.mem_get_info()[0] >= 64 * 1024 * 1024
stream = torch.cuda.Stream()
torch.manual_seed(10937)
cases = []
with torch.inference_mode(), torch.cuda.stream(stream):
    for query_length, key_length in ((128, 128), (128, 77), (129, 77)):
        q = torch.randn((1, query_length, 4, 64), device='cuda', dtype=torch.float16)
        k = torch.randn((1, key_length, 4, 64), device='cuda', dtype=torch.float16)
        v = torch.randn_like(k)
        output, lse = module.fwd(q, k, v, 0.125, False)
        reference = F.scaled_dot_product_attention(
            q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
            dropout_p=0.0, is_causal=False).transpose(1, 2)
        stream.synchronize()
        assert output.shape == q.shape and output.dtype == q.dtype and output.device == q.device
        assert lse.shape == (1, 4, query_length)
        assert torch.isfinite(output).all().item() and torch.isfinite(lse).all().item()
        assert torch.allclose(output, reference, rtol=0.01, atol=0.002), 'Tiny numeric preflight failed'
        difference = (output.float() - reference.float()).abs()
        cases.append(dict(query_length=query_length, key_length=key_length, head_dim=64,
                          max_abs_error=difference.max().item(),
                          mean_abs_error=difference.mean().item()))
torch.cuda.synchronize()
print(json.dumps(dict(preflight_passed=True, preflight_cases=cases)))
"""
    try:
        result = json.loads(_run([sys.executable, "-c", program, str(binary)], env=env, timeout=60))
    except subprocess.CalledProcessError as error:
        detail = (error.stderr or error.stdout or "No subprocess diagnostic")[-4000:]
        log_path = T4_FLASH_ROOT / "build.log"
        with log_path.open("a", encoding="utf-8") as output:
            output.write("\nTiny numeric/device/stream preflight failed:\n" + detail + "\n")
        raise RuntimeError(f"T4 attention tiny numeric/device/stream preflight failed; see {log_path}") from None
    if (not isinstance(result, dict) or result.get("preflight_passed") is not True or
            not isinstance(result.get("preflight_cases"), list) or
            len(result["preflight_cases"]) != 3):
        raise RuntimeError("T4 attention tiny preflight did not return valid results")
    return result


def _current_stamp(torch, env):
    if not T4_FLASH_STAMP.is_file() or T4_FLASH_STAMP.is_symlink():
        return None
    try:
        if T4_FLASH_STAMP.stat().st_size > 32 * 1024:
            return None
        stamp = json.loads(T4_FLASH_STAMP.read_text(encoding="utf-8"))
        if not isinstance(stamp, dict):
            return None
        expected = {
            "schema": 1, "commit": T4_FLASH_COMMIT, "patch": T4_FLASH_PATCH,
            "module": T4_FLASH_MODULE, "torch_version": str(torch.__version__),
            "torch_cuda": str(torch.version.cuda),
            "python_cache_tag": sys.implementation.cache_tag,
            "runtime_dir": str(T4_FLASH_RUNTIME),
            "preflight_passed": True,
        }
        if any(stamp.get(key) != value for key, value in expected.items()):
            return None
        binary = Path(stamp["module_file"])
        if (not binary.is_file() or binary.is_symlink() or
                binary.resolve().parent != T4_FLASH_RUNTIME.resolve() or
                not binary.name.startswith(T4_FLASH_MODULE + ".") or
                binary.suffix != ".so" or _sha256(binary) != stamp.get("module_sha256")):
            return None
        _validate_native(binary, env)
        return stamp
    except (OSError, ValueError, KeyError, RuntimeError, subprocess.SubprocessError):
        return None


def install_t4_flash_attention(comfy_root=None, *, max_jobs=2, build_timeout_s=600,
                               reserve_disk_gb=8):
    """Build the optional isolated kernel; unsupported/failed installs return None.

    `comfy_root` is accepted for notebook callers but no ComfyUI source is changed.
    Compile concurrency is capped at two processes and build time at ten minutes.
    The caller should keep ordinary SDPA when this function returns None.
    """
    try:
        import torch

        if sys.platform != "linux" or not Path("/content").is_dir():
            raise RuntimeError("This optional installer requires a Linux Colab runtime")
        if not torch.cuda.is_available() or torch.cuda.get_device_capability() != (7, 5):
            raise RuntimeError("T4 attention requires an available SM7.5 CUDA GPU")
        if not torch.version.cuda or str(torch.version.cuda).split(".")[0] != "12":
            raise RuntimeError("Audited T4 source requires PyTorch CUDA12")
        if not shutil.which("git"):
            raise RuntimeError("git is unavailable")

        # All writes remain in an owned local directory, never Drive/site-packages.
        if T4_FLASH_ROOT.is_symlink() or (T4_FLASH_ROOT.exists() and
                T4_FLASH_ROOT.resolve() != Path("/content").resolve() / T4_FLASH_ROOT.name):
            raise RuntimeError("T4 attention installation directory is not an owned local path")
        T4_FLASH_ROOT.mkdir(parents=True, exist_ok=True)
        if T4_FLASH_RUNTIME.is_symlink():
            raise RuntimeError("T4 attention runtime directory must not be a symlink")
        env = dict(os.environ)
        stamp = _current_stamp(torch, env)
        if stamp is not None:
            print("[phone-studio-t4-flash] Reusing validated pinned forward kernel")
            return stamp

        nvcc = shutil.which("nvcc")
        if not nvcc:
            from torch.utils.cpp_extension import CUDA_HOME
            candidate = Path(CUDA_HOME or "/missing") / "bin/nvcc"
            nvcc = str(candidate) if candidate.is_file() else None
        if not nvcc:
            raise RuntimeError("CUDA12 nvcc is unavailable; existing CUDA packages are preserved")
        nvcc_output = _run([nvcc, "--version"], timeout=15)
        release = re.search(r"release\s+(\d+)\.(\d+)", nvcc_output)
        if release is None or release.group(1) != str(torch.version.cuda).split(".")[0]:
            raise RuntimeError("nvcc/PyTorch CUDA major versions differ")
        if _host_available_bytes() < 4 * _GIB:
            raise RuntimeError("Need at least 4GiB available host RAM to build T4 attention")
        reserve_disk_gb = max(0.0, float(reserve_disk_gb))
        if not reserve_disk_gb < 1024:
            raise RuntimeError("Invalid local disk reserve")
        if shutil.disk_usage(T4_FLASH_ROOT).free < (reserve_disk_gb + 2) * _GIB:
            raise RuntimeError(f"Need {reserve_disk_gb:g}GiB reserve plus 2GiB build scratch on local disk")

        # Reject any old stamp before attempting a replacement build.
        T4_FLASH_STAMP.unlink(missing_ok=True)
        max_jobs = min(2, max(1, int(max_jobs)))
        build_timeout_s = min(600, max(60, int(build_timeout_s)))
        env["MAX_JOBS"] = str(max_jobs)
        env["TORCH_CUDA_ARCH_LIST"] = "7.5"
        env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        log_path = T4_FLASH_ROOT / "build.log"
        log_path.write_text(f"Pinned source {T4_FLASH_COMMIT}; patch {T4_FLASH_PATCH}\n",
                            encoding="utf-8")
        print(f"[phone-studio-t4-flash] Building optional T4 kernel; max {max_jobs} jobs, "
              f"{build_timeout_s}s build limit. Log: {log_path}")

        # Missing build helpers are isolated too; do not install requirements.txt.
        dependency_specs = {
            "ninja": "ninja>=1.11,<2", "wheel": "wheel>=0.40,<1",
            "setuptools": "setuptools>=64,<81",
        }
        missing = [spec for name, spec in dependency_specs.items()
                   if importlib.util.find_spec(name) is None]
        if missing:
            deps = T4_FLASH_ROOT / "build_deps"
            if deps.is_symlink():
                raise RuntimeError("T4 attention build-dependency directory must not be a symlink")
            _run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-input",
                  "--target", str(deps), *missing], env=env, timeout=120, log_path=log_path)
            env["PYTHONPATH"] = str(deps) + os.pathsep + env.get("PYTHONPATH", "")
            env["PATH"] = str(deps / "bin") + os.pathsep + env.get("PATH", "")

        with tempfile.TemporaryDirectory(prefix="build-", dir=T4_FLASH_ROOT) as scratch:
            source_dir = Path(scratch) / "source"
            candidate_runtime = Path(scratch) / "runtime"
            source_dir.mkdir()
            _run(["git", "init", "--quiet"], cwd=source_dir, timeout=30)
            _run(["git", "remote", "add", "origin",
                  "https://github.com/ssiu/flash-attention-turing.git"], cwd=source_dir)
            _run(["git", "-c", "core.autocrlf=false", "fetch", "--depth", "1",
                  "origin", T4_FLASH_COMMIT], cwd=source_dir, timeout=120, log_path=log_path)
            _run(["git", "-c", "core.autocrlf=false", "checkout", "--detach", "FETCH_HEAD"],
                 cwd=source_dir, timeout=60, log_path=log_path)
            if _run(["git", "rev-parse", "HEAD"], cwd=source_dir) != T4_FLASH_COMMIT:
                raise RuntimeError("T4 source revision mismatch")
            _run(["git", "submodule", "update", "--init", "--depth", "1", "csrc/cutlass"],
                 cwd=source_dir, timeout=180, log_path=log_path)
            if _run(["git", "rev-parse", "HEAD"], cwd=source_dir / "csrc/cutlass") != T4_FLASH_CUTLASS_COMMIT:
                raise RuntimeError("CUTLASS submodule revision mismatch")
            _patch_source(source_dir)
            _run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation",
                  "--no-input", "--no-cache-dir", "--target", str(candidate_runtime), str(source_dir)],
                 env=env, timeout=build_timeout_s, log_path=log_path)
            binaries = list(candidate_runtime.glob(T4_FLASH_MODULE + ".*.so"))
            if len(binaries) != 1 or binaries[0].is_symlink():
                raise RuntimeError("Isolated T4 build did not produce exactly one native module")
            preflight = _validate_native(binaries[0], env)
            if T4_FLASH_RUNTIME.exists():
                # The only removal is this fixed, verified, installer-owned local path.
                if T4_FLASH_RUNTIME.resolve() != T4_FLASH_ROOT.resolve() / "runtime":
                    raise RuntimeError("Refusing to replace an unowned runtime directory")
                shutil.rmtree(T4_FLASH_RUNTIME)
            candidate_runtime.rename(T4_FLASH_RUNTIME)

        binary = next(T4_FLASH_RUNTIME.glob(T4_FLASH_MODULE + ".*.so"))
        stamp = {
            "schema": 1, "commit": T4_FLASH_COMMIT, "patch": T4_FLASH_PATCH,
            "module": T4_FLASH_MODULE, "torch_version": str(torch.__version__),
            "torch_cuda": str(torch.version.cuda),
            "python_cache_tag": sys.implementation.cache_tag,
            "runtime_dir": str(T4_FLASH_RUNTIME), "module_file": str(binary),
            "module_sha256": _sha256(binary), "nvcc_release": release.group(0),
            "preflight_passed": True,
            "preflight_cases": preflight["preflight_cases"],
            "built_at_unix": int(time.time()),
        }
        temporary_stamp = T4_FLASH_ROOT / "install.json.part"
        temporary_stamp.write_text(json.dumps(stamp, indent=2) + "\n", encoding="utf-8")
        temporary_stamp.replace(T4_FLASH_STAMP)
        print("[phone-studio-t4-flash] Validated pinned forward kernel ready; "
              "speed/quality remain unmeasured")
        return stamp
    except Exception as error:
        print(f"[phone-studio-t4-flash] Optional kernel unavailable: {type(error).__name__}: {error}. "
              "Keeping ordinary PyTorch attention.")
        return None

'''
        _t4_installer_namespace = {'__name__': 'phone_studio_t4_flash_installer'}
        exec(compile(T4_FLASH_INSTALLER_CODE, 'phone_studio_t4_flash_installer', 'exec'),
             _t4_installer_namespace)
        _t4_manifest = _t4_installer_namespace['install_t4_flash_attention'](
            RUNTIME_ROOT, reserve_disk_gb=MODEL_CACHE_RESERVE_GB)
        T4_FLASH_ATTENTION_AVAILABLE = _t4_manifest is not None
        print('T4 Fast attention package ready:', T4_FLASH_ATTENTION_AVAILABLE,
              '| Bridge validates the ComfyUI model hook at server startup.')
# --- End optional T4 Fast attention installer ---

print(f'Local model cache: {"on" if LOCAL_MODEL_CACHE else "off"}; maximum {MODEL_CACHE_MAX_GB} GiB.')
print(f'VRAM mode: {VRAM_MODE}; automatic compilation: {PHONE_STUDIO_AUTO_COMPILE == "1"}.')

# Default checkpoint: used straight from Drive when it is already there. If it is
# missing, it is downloaded once from DEFAULT_CHECKPOINT_URL into Drive (kept for
# every later session).
# Exact checkpoint name, relative to Drive's models/checkpoints folder.
# Include the subfolder prefix here if you move it into a subfolder.
DEFAULT_CHECKPOINT = 'waiIllustriousSDXL_v170.safetensors'
# Where to get the default checkpoint when it is not in Drive. Civitai may ask for a
# login: if the download is refused, add CIVITAI_API_KEY in Colab Secrets and rerun.
# Set this to '' to turn the automatic download off (a missing model is then an error).
DEFAULT_CHECKPOINT_URL = 'https://civitai.red/api/download/models/2883731?fileId=2763986'
# File name used only if DEFAULT_CHECKPOINT is empty and the server sends no file name.
DEFAULT_CHECKPOINT_FALLBACK_NAME = 'waiIllustriousSDXL_v170.safetensors'
WARMUP_ON_START = True
COMPLETED_WARMUP_FILE = Path('/content/phone_studio_startup_warmup.json')  # Ephemeral, instance-bound proof.
WARMUP_WIDTH = 768  # Match the phone default so first Tall render reuses warmed shapes.
WARMUP_HEIGHT = 1024
WARMUP_STEPS = 2
WARMUP_SAMPLER = 'dpmpp_2m'
WARMUP_SCHEDULER = 'karras'
WARMUP_TIMEOUT_SECONDS = 900
WARMUP_PROMPT = 'a quiet mountain lake at sunrise, landscape, anime illustration'


def _download_default_checkpoint_unlocked(checkpoint_root, url, requested=''):
    """Download the default checkpoint into Drive and return its name relative to
    checkpoint_root. Writes to a .part file first, so an interrupted download never
    leaves a half-written model that looks usable."""
    import json
    import re
    import struct
    import time
    from urllib.parse import unquote, urlparse

    import requests

    suffixes = {'.safetensors', '.ckpt', '.pt'}
    root = checkpoint_root.resolve()
    if requested and Path(requested).suffix.lower() not in suffixes:
        raise ValueError(f'DEFAULT_CHECKPOINT must end in .safetensors, .ckpt or .pt: {requested}')

    # Send the Civitai key to Civitai only, never to any other host.
    headers = {}
    host = (urlparse(url).hostname or '').lower()
    if CIVITAI_API_KEY and re.search(r'(^|\.)civitai\.(com|red)$', host):
        headers['Authorization'] = f'Bearer {CIVITAI_API_KEY}'
    key_hint = ('Check that CIVITAI_API_KEY in Colab Secrets is valid, then rerun.' if CIVITAI_API_KEY
                else 'Add CIVITAI_API_KEY in Colab Secrets (create one in your Civitai account '
                     'settings), then rerun.')

    def looks_like_safetensors(path):
        try:
            size = path.stat().st_size
            with open(path, 'rb') as handle:
                (header_length,) = struct.unpack('<Q', handle.read(8))
                if not 0 < header_length <= 100 * 1024**2 or 8 + header_length > size:
                    return False
                json.loads(handle.read(header_length))
            return True
        except (OSError, ValueError, struct.error):
            return False

    for attempt in range(1, 4):
        temp = None
        try:
            with requests.get(url, headers=headers, stream=True, allow_redirects=True,
                              timeout=(30, 120)) as response:
                if response.status_code in (401, 403):
                    raise PermissionError(f'Civitai refused the download (HTTP {response.status_code}). {key_hint}')
                response.raise_for_status()
                content_type = response.headers.get('content-type', '').lower()
                if content_type.startswith('text/') or 'json' in content_type:
                    raise RuntimeError(f'The download link returned a web page, not a model file. {key_hint}')

                # Decide the file name before writing anything.
                if requested:
                    relative = requested
                else:
                    disposition = response.headers.get('content-disposition', '')
                    match = (re.search(r"filename\*\s*=\s*(?:UTF-8'')?([^;]+)", disposition, re.I)
                             or re.search(r'filename\s*=\s*"?([^";]+)"?', disposition, re.I))
                    served = Path(unquote(match.group(1).strip().strip('"'))).name if match else ''
                    relative = re.sub(r'[^A-Za-z0-9._ -]+', '_', served).strip(' .')
                    if Path(relative).suffix.lower() not in suffixes:
                        relative = DEFAULT_CHECKPOINT_FALLBACK_NAME
                destination = (root / relative).resolve()
                try:
                    relative = destination.relative_to(root).as_posix()
                except ValueError:
                    raise ValueError(f'DEFAULT_CHECKPOINT must stay inside {root}: {requested}') from None
                if destination.is_file():
                    print(f'{relative} is already in Drive; using it without downloading.')
                    return relative

                destination.parent.mkdir(parents=True, exist_ok=True)
                temp = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.part')
                total = int(response.headers.get('content-length') or 0)
                size_note = f' ({total / 1024**3:.2f} GiB)' if total else ''
                print(f'Downloading {relative}{size_note} into Drive. This is a one-time download...')
                downloaded = 0
                started = last_report = time.monotonic()
                with open(temp, 'wb') as handle:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if not chunk:
                            continue
                        handle.write(chunk)
                        downloaded += len(chunk)
                        now = time.monotonic()
                        if now - last_report >= 10:
                            percent = f' ({downloaded * 100 / total:.0f}%)' if total else ''
                            speed = downloaded / (now - started) / 1024**2
                            print(f'  {downloaded / 1024**3:.2f} GiB{percent} at {speed:.0f} MiB/s')
                            last_report = now
                if total and not response.headers.get('content-encoding') and downloaded != total:
                    raise requests.exceptions.ChunkedEncodingError(
                        f'incomplete download ({downloaded} of {total} bytes)')

            if destination.suffix.lower() == '.safetensors' and not looks_like_safetensors(temp):
                raise RuntimeError('The downloaded file is not a valid SafeTensors model '
                                   f'(the link may need a login). {key_hint}')
            temp.replace(destination)
            print(f'Downloaded {relative} ({downloaded / 1024**3:.2f} GiB) in '
                  f'{time.monotonic() - started:.0f}s. It stays in Drive for future sessions.')
            return relative
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout,
                requests.exceptions.ChunkedEncodingError) as exc:
            if attempt == 3:
                raise
            print(f'Download interrupted ({exc}); retrying ({attempt}/2)...')
            time.sleep(5 * attempt)
        finally:
            if temp is not None:
                temp.unlink(missing_ok=True)


def download_default_checkpoint(checkpoint_root, url, requested=''):
    # Shared Drive writers serialize initial preparation; healthy instances are
    # untouched. An occupied lock fails finitely instead of corrupting a .part.
    with _worker_setup['exclusive_artifact'](checkpoint_root / '.phone-studio-default-checkpoint.lock'):
        return _download_default_checkpoint_unlocked(checkpoint_root, url, requested)

def resolve_default_checkpoint(checkpoint_root, requested='', download_url=''):
    import re
    available = sorted(
        item.relative_to(checkpoint_root).as_posix()
        for item in checkpoint_root.rglob('*')
        if item.is_file() and item.suffix.lower() in {'.safetensors', '.ckpt', '.pt'}
    )

    def download_or_raise(problem, advice):
        if not download_url:
            raise FileNotFoundError(f'{problem} {advice}')
        print(problem)
        print('Fetching it from DEFAULT_CHECKPOINT_URL...')
        return download_default_checkpoint(checkpoint_root, download_url, requested)

    if requested:
        if requested in available:
            return requested
        return download_or_raise(
            f'Default checkpoint is not in {checkpoint_root}: {requested}',
            'Put it there, or set DEFAULT_CHECKPOINT_URL to download it automatically.')
    matches = []
    for name in available:
        normalized = re.sub(r'[^a-z0-9]', '', Path(name).stem.lower())
        if ('wai' in normalized and 'illustrious' in normalized
                and re.search(r'v170(?![0-9])', normalized)):
            matches.append(name)
    if len(matches) == 1:
        return matches[0]
    if not matches:
        return download_or_raise(
            f'Could not identify WAI Illustrious v170 in {checkpoint_root}.',
            'Set DEFAULT_CHECKPOINT to its exact filename above, or set '
            f'DEFAULT_CHECKPOINT_URL to download it. Available checkpoints: {available}')
    raise ValueError(
        'Several WAI Illustrious v170 files were found. '
        f'Set DEFAULT_CHECKPOINT to one of these exact names: {matches}'
    )

DEFAULT_CHECKPOINT = resolve_default_checkpoint(MODELS_ROOT / 'checkpoints', DEFAULT_CHECKPOINT, DEFAULT_CHECKPOINT_URL)
print('Default checkpoint:', DEFAULT_CHECKPOINT)
print(f'Startup warm-up: {WARMUP_ON_START} ({WARMUP_WIDTH} x {WARMUP_HEIGHT}, {WARMUP_STEPS} steps).')


print('Fast denoise cache:', FAST_DENOISE_CACHE, '| Local output cache:', LOCAL_OUTPUT_CACHE)


In [ ]:
# Phone Studio uses the same two tuned presets for generation and editing.
print('Phone presets: Balanced 24 steps / Fast 18 steps; CFG 6.5, dpmpp_2m / karras.')
# Self-contained optional edit setup; the helper is not executed during source checks.
EDIT_WORKER_SETUP_CODE = r'''
"""Optional pinned Fooocus auxiliary setup, invoked by the worker notebook.

Importing this helper installs/runs nothing. No Torch/CUDA dependency is replaced.
"""
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import tempfile
import time
import urllib.request
import uuid
from pathlib import Path

FOOOCUS_SOURCE_REVISION = "bd6d8fd5104d539187dfb172d07352215c09fcaa"
FOOOCUS_WEIGHTS_REVISION = "74bbcc070e55219adb9b6c3b0d035b34e3697d1d"
FOOOCUS_WEIGHTS = {
    "fooocus_inpaint_head.pth": (52602, "32f7f838e0c6d8f13437ba8411e77a4688d77a2e34df8857e4ef4d51f6b97692"),
    "inpaint_v26.fooocus.patch": (1323362033, "f8657a025104e22d70f9c060635d8e8c2196f433871a2f68dc40abd2171f0d59"),
}


def _hash_file(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()


def _download_weight(name: str, root: Path, reserve_gb: float) -> None:
    size, digest = FOOOCUS_WEIGHTS[name]
    path = root / name
    if path.is_file() and path.stat().st_size == size and _hash_file(path) == digest:
        return
    if path.exists():
        raise RuntimeError(f"Existing {name} does not match the pinned publisher file; move it aside before enabling Fooocus.")
    if shutil.disk_usage(root).free < size + reserve_gb * 1024**3:
        raise RuntimeError("Not enough free space for the Fooocus auxiliary weights and the configured disk reserve.")
    url = f"https://huggingface.co/lllyasviel/fooocus_inpaint/resolve/{FOOOCUS_WEIGHTS_REVISION}/{name}"
    temporary = root / (name + "." + uuid.uuid4().hex + ".part")
    started = time.monotonic()
    hasher, downloaded = hashlib.sha256(), 0
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "PhoneStudio-edit-worker"})
        with urllib.request.urlopen(request, timeout=45) as response, temporary.open("xb") as handle:
            while block := response.read(4 * 1024 * 1024):
                downloaded += len(block)
                if downloaded > size or time.monotonic() - started > 600:
                    raise RuntimeError("Fooocus download exceeded its expected size/time limit.")
                hasher.update(block)
                handle.write(block)
        if downloaded != size or hasher.hexdigest() != digest:
            raise RuntimeError("Fooocus publisher-file integrity check failed.")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _install_source(custom_nodes_root: Path) -> Path:
    target = custom_nodes_root / "phone_studio_fooocus_inpaint"
    if target.exists():
        try:
            revision = subprocess.check_output(["git", "-C", str(target), "rev-parse", "HEAD"], text=True, timeout=15).strip()
            changes = subprocess.check_output(["git", "-C", str(target), "status", "--porcelain", "--untracked-files=no"], text=True, timeout=15)
        except (subprocess.SubprocessError, OSError) as error:
            raise RuntimeError("The managed Fooocus folder is not the pinned checkout; preserve it and move it aside before retrying.") from error
        if revision != FOOOCUS_SOURCE_REVISION or changes.strip():
            raise RuntimeError("The managed Fooocus checkout was changed; preserve it and move it aside before retrying.")
        return target
    temporary = Path(tempfile.mkdtemp(prefix="phone_studio_fooocus_", dir=custom_nodes_root))
    try:
        subprocess.run(["git", "init", "-q", str(temporary)], check=True, timeout=30)
        subprocess.run(["git", "-C", str(temporary), "remote", "add", "origin", "https://github.com/Acly/comfyui-inpaint-nodes.git"], check=True, timeout=30)
        subprocess.run(["git", "-C", str(temporary), "fetch", "-q", "--depth", "1", "origin", FOOOCUS_SOURCE_REVISION], check=True, timeout=90)
        subprocess.run(["git", "-C", str(temporary), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True, timeout=30)
        revision = subprocess.check_output(["git", "-C", str(temporary), "rev-parse", "HEAD"], text=True, timeout=15).strip()
        if revision != FOOOCUS_SOURCE_REVISION:
            raise RuntimeError("Fooocus source revision verification failed.")
        os.replace(temporary, target)
        return target
    finally:
        if temporary.exists():
            # This is our freshly created, confined temporary checkout only.
            shutil.rmtree(temporary)


def setup_worker_editing(custom_nodes_root: str | Path, models_root: str | Path,
                         enabled: bool = True, reserve_gb: float = 8) -> dict:
    """Request the free auxiliary by default; failure keeps honest stock edits.

    Comfy's existing dependencies must supply Torch/NumPy/Kornia. No package
    resolver is allowed to upgrade these or install a different CUDA runtime.
    """
    result = {"requested": bool(enabled), "ready": False, "sourceRevision": FOOOCUS_SOURCE_REVISION,
              "weightsRevision": FOOOCUS_WEIGHTS_REVISION,
              "message": "Stock SDXL editing is available; Fooocus is disabled."}
    if not enabled:
        return result
    try:
        for module in ("torch", "numpy", "kornia"):
            if importlib.util.find_spec(module) is None:
                raise RuntimeError(f"Fooocus needs the existing ComfyUI {module} dependency; it was not replaced or installed automatically.")
        custom_nodes_root, models_root = Path(custom_nodes_root).resolve(), Path(models_root).resolve()
        custom_nodes_root.mkdir(parents=True, exist_ok=True)
        model_dir = models_root / "inpaint"
        model_dir.mkdir(parents=True, exist_ok=True)
        # Install a source only after the required weights are verified. A
        # failed download must not leave an active partial custom-node install.
        for name in FOOOCUS_WEIGHTS:
            _download_weight(name, model_dir, reserve_gb)
        source = _install_source(custom_nodes_root)
        result.update(ready=True, message="Pinned Fooocus auxiliary prepared. Start or restart ComfyUI; startup node registration determines availability.",
                      sourcePath=str(source), weights={name: digest for name, (_, digest) in FOOOCUS_WEIGHTS.items()})
    except Exception as error:
        # No credentials are used by this public-download helper.
        result["message"] = f"Fooocus unavailable: {error}. Stock masked SDXL edits remain available."
    print("[phone-studio-edit]", result["message"])
    return result


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description="Prepare pinned optional SDXL inpaint support without replacing Torch/CUDA.")
    parser.add_argument("--custom-nodes-root", required=True)
    parser.add_argument("--models-root", required=True)
    parser.add_argument("--reserve-gb", type=float, default=8)
    args = parser.parse_args()
    print(json.dumps(setup_worker_editing(args.custom_nodes_root, args.models_root, reserve_gb=args.reserve_gb)))
'''
_edit_setup = {'__name__': 'phone_studio_edit_worker_setup'}
exec(compile(EDIT_WORKER_SETUP_CODE, '<phone-studio-edit-setup>', 'exec'), _edit_setup)
FOOOCUS_SETUP = _edit_setup['setup_worker_editing'](
    CUSTOM_NODES_ROOT, MODELS_ROOT, enabled=ENABLE_FOOOCUS_INPAINT,
    reserve_gb=FOOOCUS_DRIVE_RESERVE_GB)


In [ ]:
# Add Phone Studio's small authenticated API as a ComfyUI custom node.
# It lists local models and downloads a Civitai *model version* directly to Drive.
BRIDGE_DIR = RUNTIME_CUSTOM_NODES_ROOT / 'phone_studio_bridge'
BRIDGE_DIR.mkdir(parents=True, exist_ok=True)

BRIDGE_CODE = r'''
import asyncio
import hmac
import os
import re
import uuid
from collections import OrderedDict
from pathlib import Path

import requests
import torch
from aiohttp import web
from server import PromptServer

MODELS_ROOT = Path(os.environ.get('PHONE_STUDIO_MODELS_ROOT', '/content/drive/MyDrive/PhoneStudio/models'))
PHONE_TOKEN = os.environ.get('PHONE_STUDIO_TOKEN', '')
CIVITAI_KEY = os.environ.get('CIVITAI_API_KEY', '')
COMPLETED_WARMUP_FILE = Path(os.environ.get('PHONE_STUDIO_COMPLETED_WARMUP_FILE', '/content/phone_studio_startup_warmup.json'))
_COMPLETED_WARMUP = None
_STARTUP_WARMUP_REUSE_ALLOWED = True
DEFAULT_CHECKPOINT = os.environ.get('PHONE_STUDIO_DEFAULT_CHECKPOINT', '')
DOWNLOADS = {}
INSTANCE_ID = os.environ.get('PHONE_STUDIO_EXPECTED_INSTANCE_ID') or uuid.uuid4().hex
routes = PromptServer.instance.routes

# --- Per-image speed extras and cache fingerprints ---
"""Embed in BRIDGE_CODE before optional model/save hooks; no dependencies.

The app's extra_data.phone_studio_speed_extras selects per-image extras.
Missing values keep the notebook default (on). Launch-time precision is separate.
Stock IS_CHANGED fingerprints prevent an old accelerated image being returned
from cache after the switch changes, without discarding model/text caches.
Sources: ComfyUI execution.py, comfy_execution/caching.py, nodes.py.
"""

import contextvars as _ps_speed_contextvars
import functools as _ps_speed_functools
import inspect as _ps_speed_inspect
import os as _ps_speed_os

_PS_COMPATIBILITY_MODE = _ps_speed_os.environ.get('PHONE_STUDIO_COMPATIBILITY_MODE', '0') == '1'
_PS_SPEED_EXTRAS = _ps_speed_contextvars.ContextVar('phone_studio_speed_extras', default=not _PS_COMPATIBILITY_MODE)
_PS_SPEED_SWITCH_AVAILABLE = False


def _phone_studio_speed_extras_enabled():
    return not _PS_COMPATIBILITY_MODE and _PS_SPEED_EXTRAS.get()


def _ps_is_edit_graph(prompt):
    if not isinstance(prompt, dict):
        return False
    edit_nodes = {'LoadImage', 'LoadImageMask', 'VAEEncode', 'VAEEncodeForInpaint',
                  'SetLatentNoiseMask', 'InpaintModelConditioning', 'ImagePadForOutpaint'}
    for node in prompt.values():
        if not isinstance(node, dict):
            continue
        kind = node.get('class_type', '')
        if kind in edit_nodes or (isinstance(kind, str) and kind.startswith('INPAINT_')):
            return True
        if kind == 'SaveImage' and isinstance(node.get('inputs'), dict):
            if str(node['inputs'].get('filename_prefix', '')).startswith('PhoneStudioEdit'):
                return True
    return False


def _ps_speed_requested(extra_data, prompt=None):
    if not isinstance(extra_data, dict):
        return not _PS_COMPATIBILITY_MODE and not _ps_is_edit_graph(prompt)
    value = extra_data.get('phone_studio_speed_extras', True)
    if type(value) is not bool:
        raise ValueError('phone_studio_speed_extras must be a Boolean')
    return False if _PS_COMPATIBILITY_MODE or _ps_is_edit_graph(prompt) else value


def _ps_install_speed_switch():
    global _PS_SPEED_SWITCH_AVAILABLE
    try:
        import execution as _ps_speed_execution
        import nodes as _ps_speed_nodes
        executor = _ps_speed_execution.PromptExecutor
        marker = '_phone_studio_speed_switch_v1'
        if getattr(executor.execute, marker, False):
            _PS_SPEED_SWITCH_AVAILABLE = True
            return
        hooks = []
        for name in ('execute', 'execute_async'):
            original = getattr(executor, name, None)
            if original is None:
                continue
            signature = _ps_speed_inspect.signature(original)
            if not {'self', 'prompt', 'prompt_id', 'extra_data'} <= set(signature.parameters):
                raise RuntimeError('ComfyUI executor API changed')

            def make_wrapper(fn, sig):
                def select_flag(args, kwargs):
                    bound = sig.bind(*args, **kwargs)
                    bound.apply_defaults()
                    return _ps_speed_requested(bound.arguments.get('extra_data'), bound.arguments.get('prompt'))

                if _ps_speed_inspect.iscoroutinefunction(fn):
                    @_ps_speed_functools.wraps(fn)
                    async def wrapped(*args, **kwargs):
                        token = _PS_SPEED_EXTRAS.set(select_flag(args, kwargs))
                        try:
                            return await fn(*args, **kwargs)
                        finally:
                            _PS_SPEED_EXTRAS.reset(token)
                else:
                    @_ps_speed_functools.wraps(fn)
                    def wrapped(*args, **kwargs):
                        token = _PS_SPEED_EXTRAS.set(select_flag(args, kwargs))
                        try:
                            return fn(*args, **kwargs)
                        finally:
                            _PS_SPEED_EXTRAS.reset(token)
                setattr(wrapped, marker, True)
                return wrapped

            hooks.append((name, make_wrapper(original, signature)))

        classes = (_ps_speed_nodes.KSampler, _ps_speed_nodes.SaveImage)
        if any(cls.__module__ != 'nodes' or hasattr(cls, 'IS_CHANGED') for cls in classes):
            raise RuntimeError('stock sampler/save fingerprint API is unavailable or already customized')

        # Only a cheap Boolean cache fingerprint changes. Standard sampling,
        # model/CLIP caches, graph fields and class execution methods stay intact.
        def fingerprint(**inputs):
            return _phone_studio_speed_extras_enabled()

        for cls in classes:
            cls.IS_CHANGED = staticmethod(fingerprint)
        for name, wrapped in hooks:
            setattr(executor, name, wrapped)
        _PS_SPEED_SWITCH_AVAILABLE = True
        print('[phone-studio] Per-image speed switch ready; sampling/save caches distinguish on/off.')
    except Exception as error:
        print(f'[phone-studio] Per-image speed switch unavailable ({error}); notebook defaults retained.')


_ps_install_speed_switch()


@web.middleware
async def _ps_speed_switch_guard(request, handler):
    if request.method == 'POST' and request.path == '/prompt':
        try:
            body = await request.json()
        except Exception:
            body = None  # Standard handler reports malformed request JSON.
        extra = body.get('extra_data') if isinstance(body, dict) else None
        if isinstance(extra, dict) and 'phone_studio_speed_extras' in extra:
            value = extra['phone_studio_speed_extras']
            message = None
            if type(value) is not bool:
                message = 'The extra acceleration setting must be true or false.'
            elif not _PS_SPEED_SWITCH_AVAILABLE:
                message = 'This ComfyUI version cannot switch image acceleration. Update/rerun the notebook bridge.'
            if message:
                return web.json_response({
                    'error': {'type': 'phone_studio_speed_switch_unavailable',
                              'message': message, 'details': message, 'extra_info': {}},
                    'node_errors': {},
                }, status=400)
    return await handler(request)


PromptServer.instance.app.middlewares.append(_ps_speed_switch_guard)
# --- End per-image speed extras ---

# --- Read model weights from Colab's local disk after one sequential copy ---
# Keep model discovery and Civitai downloads on Drive. Intercept only the
# standard weight loader, so checkpoint/LoRA names and workflow APIs stay intact.
import functools
import hashlib
import shutil
import threading
import time
import comfy.utils

CACHE_ROOT = Path(os.environ.get('PHONE_STUDIO_MODEL_CACHE_ROOT', '/content/phone_studio_model_cache'))
CACHE_ENABLED = os.environ.get('PHONE_STUDIO_LOCAL_MODEL_CACHE', '1') == '1'
CACHE_MAX_BYTES = int(float(os.environ.get('PHONE_STUDIO_MODEL_CACHE_MAX_GB', '24')) * 1024**3)
CACHE_RESERVE_BYTES = int(float(os.environ.get('PHONE_STUDIO_MODEL_CACHE_RESERVE_GB', '8')) * 1024**3)
_CACHE_LOCK = threading.Lock()


def local_model_path(filename):
    if not CACHE_ENABLED:
        return filename
    temp = None
    try:
        source = Path(filename).resolve()
        # Do not cache arbitrary uploads, outputs, or files outside the model store.
        source.relative_to(MODELS_ROOT.resolve())
        if source.suffix.lower() not in {'.safetensors', '.sft', '.ckpt', '.pt', '.pth', '.bin'}:
            return filename
        with _CACHE_LOCK:
            stat = source.stat()
            # A replacement model gets a different local file, even if its name
            # stays the same. Never overwrite a file that may be memory-mapped.
            signature = f'{source}|{stat.st_size}|{stat.st_mtime_ns}'
            key = hashlib.sha256(signature.encode()).hexdigest()
            destination = CACHE_ROOT / key / source.name
            if destination.is_file() and destination.stat().st_size == stat.st_size:
                return str(destination)
            CACHE_ROOT.mkdir(parents=True, exist_ok=True)
            used = sum(item.stat().st_size for item in CACHE_ROOT.rglob('*') if item.is_file())
            free = shutil.disk_usage(CACHE_ROOT).free
            if used + stat.st_size > CACHE_MAX_BYTES or free - stat.st_size < CACHE_RESERVE_BYTES:
                print(f'[phone-studio] Local cache full/low disk; loading {source.name} from Drive.')
                return filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            temp = destination.with_name(destination.name + f'.{uuid.uuid4().hex}.part')
            started = time.perf_counter()
            print(f'[phone-studio] Caching {source.name} ({stat.st_size / 1024**3:.2f} GiB) on local disk...')
            shutil.copyfile(source, temp)
            after = source.stat()
            if (after.st_size, after.st_mtime_ns) != (stat.st_size, stat.st_mtime_ns):
                raise OSError('Source model changed during copying; retry on the next load.')
            if temp.stat().st_size != stat.st_size:
                raise OSError('Incomplete model cache copy.')
            temp.replace(destination)
            print(f'[phone-studio] Local copy ready in {time.perf_counter() - started:.1f}s; reused on later loads.')
            return str(destination)
    except ValueError:
        return filename  # Model is outside the persistent models directory.
    except OSError as exc:
        print(f'[phone-studio] Local cache unavailable ({exc}); using the original model file.')
        return filename
    finally:
        if temp is not None:
            try:
                temp.unlink(missing_ok=True)
            except OSError:
                pass


_drive_load_torch_file = comfy.utils.load_torch_file

@functools.wraps(_drive_load_torch_file)
def _load_torch_file_cached(ckpt, *args, **kwargs):
    return _drive_load_torch_file(local_model_path(ckpt), *args, **kwargs)

comfy.utils.load_torch_file = _load_torch_file_cached
print(f'[phone-studio] Local model cache {"enabled" if CACHE_ENABLED else "disabled"}.')

# --- Optional compilation for newer supported GPUs, disabled on T4 ---------
# Use ComfyUI's sample-time wrapper to preserve normal parameter names for
# LoRA loading/patching. Never permanently replace diffusion_model itself.
if os.environ.get('PHONE_STUDIO_AUTO_COMPILE', '0') == '1' and not _PS_COMPATIBILITY_MODE and torch.cuda.is_available():
    if torch.cuda.get_device_capability(0)[0] < 8:
        print('[phone-studio] Skipping torch.compile on T4/pre-Ampere (current Triton support starts at 8.0).')
    else:
        try:
            import torch._dynamo
            import comfy.sd
            from comfy_api.torch_helpers import set_torch_compile_wrapper

            torch._dynamo.config.suppress_errors = True
            _original_load_checkpoint = comfy.sd.load_checkpoint_guess_config

            @functools.wraps(_original_load_checkpoint)
            def _load_checkpoint_and_compile(*args, **kwargs):
                result = _original_load_checkpoint(*args, **kwargs)
                if not result or result[0] is None:
                    return result
                try:
                    compiled_patcher = result[0].clone(disable_dynamic=True)
                    set_torch_compile_wrapper(
                        model=compiled_patcher, backend='inductor', mode='default',
                        fullgraph=False, dynamic=None,
                    )
                    print('[phone-studio] Optional sample-time compilation enabled; first sample may be slow.')
                    return (compiled_patcher, *result[1:])
                except Exception as exc:
                    print(f'[phone-studio] Compilation setup skipped: {exc}')
                    return result

            comfy.sd.load_checkpoint_guess_config = _load_checkpoint_and_compile
        except Exception as exc:
            print(f'[phone-studio] Compilation helper unavailable ({exc}); using normal execution.')

# --- Optional T4 Fast attention bridge hook ---
"""Optional bridge snippet for the pinned, patched T4 attention extension.

Embed this source in BRIDGE_CODE after the existing checkpoint-load wrappers.
It is inactive unless PHONE_STUDIO_T4_FLASH_ATTENTION=1. The installer must
produce a matching manifest; an ordinary flash-attention-turing install is not
accepted. No PyTorch operator or imported ComfyUI attention reference is patched.

Sources inspected:
https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/ldm/modules/attention.py
https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/model_patcher.py
https://github.com/ssiu/flash-attention-turing/tree/9ef98fcb506bb1e2fe3cece50935e2935bf6b124
"""


def _ps_install_t4_flash_attention():
    import functools as _t4_functools
    import hashlib as _t4_hashlib
    import importlib.util as _t4_importlib
    import inspect as _t4_inspect
    import json as _t4_json
    import logging as _t4_logging
    import math as _t4_math
    import os as _t4_os
    from pathlib import Path as _T4_Path
    import sys as _t4_sys

    if _t4_os.environ.get('PHONE_STUDIO_T4_FLASH_ATTENTION', '0') != '1':
        return

    _t4_log = _t4_logging.getLogger('phone-studio.t4-attention')
    _t4_commit = '9ef98fcb506bb1e2fe3cece50935e2935bf6b124'
    _t4_patch = 'phone-studio-t4-forward-v1'
    _t4_module_name = 'phone_studio_flash_attn_turing'
    _t4_root = _T4_Path('/content/phone_studio_t4_flash')
    _t4_runtime = _t4_root / 'runtime'
    _t4_stamp = _t4_root / 'install.json'

    try:
        import torch as _t4_torch
        import comfy.sd as _t4_sd
        import comfy.model_base as _t4_model_base
        from comfy.ldm.modules import attention as _t4_attention

        if not _t4_torch.cuda.is_available():
            raise RuntimeError('CUDA is unavailable')
        if _t4_torch.cuda.get_device_capability(0) != (7, 5):
            raise RuntimeError('this extension is restricted to T4/Turing SM7.5')
        if (_t4_root.is_symlink() or _t4_runtime.is_symlink()
                or _t4_stamp.is_symlink() or not _t4_stamp.is_file()
                or _t4_stamp.stat().st_size > 65536):
            raise RuntimeError('a matching local installer manifest is required')
        _t4_manifest = _t4_json.loads(_t4_stamp.read_text(encoding='utf-8'))
        _t4_identity = {
            'schema': 1,
            'commit': _t4_commit,
            'patch': _t4_patch,
            'module': _t4_module_name,
            'torch_version': str(_t4_torch.__version__),
            'torch_cuda': str(_t4_torch.version.cuda),
            'python_cache_tag': _t4_sys.implementation.cache_tag,
            'runtime_dir': str(_t4_runtime),
            'preflight_passed': True,
        }
        if not isinstance(_t4_manifest, dict) or any(
                _t4_manifest.get(key) != value
                for key, value in _t4_identity.items()):
            raise RuntimeError('installer manifest does not match this runtime/build')
        _t4_binary = _T4_Path(_t4_manifest.get('module_file', ''))
        if (not _t4_binary.is_absolute() or _t4_binary.is_symlink()
                or _t4_binary.parent != _t4_runtime
                or _t4_binary.resolve().parent != _t4_runtime.resolve()
                or not _t4_binary.name.startswith(_t4_module_name + '.')
                or _t4_binary.suffix != '.so' or not _t4_binary.is_file()):
            raise RuntimeError('installer native module path is invalid')
        _t4_digest = _t4_hashlib.sha256()
        with _t4_binary.open('rb') as _t4_binary_file:
            for _t4_chunk in iter(lambda: _t4_binary_file.read(1024 * 1024), b''):
                _t4_digest.update(_t4_chunk)
        if _t4_digest.hexdigest() != _t4_manifest.get('module_sha256'):
            raise RuntimeError('installer native module hash has changed')

        # Fail closed on an upstream hook/signature change. The documented
        # per-model override is read inside existing decorated attention calls,
        # including references imported before this custom bridge was loaded.
        _t4_default = _t4_attention.attention_pytorch
        _t4_default_raw = _t4_inspect.unwrap(_t4_default)
        _t4_required = ('q', 'k', 'v', 'heads', 'mask', 'attn_precision',
                        'skip_reshape', 'skip_output_reshape')
        if tuple(_t4_inspect.signature(_t4_default_raw).parameters)[:8] != _t4_required:
            raise RuntimeError('ComfyUI attention signature is not supported')
        if ('optimized_attention_override' not in
                _t4_inspect.getsource(_t4_attention.wrap_attn)):
            raise RuntimeError('ComfyUI per-model attention override is unavailable')
        if (_t4_attention.optimized_attention is not _t4_default
                or _t4_attention.optimized_attention_masked is not _t4_default):
            raise RuntimeError('the ordinary PyTorch attention baseline is required')
        if not hasattr(_t4_model_base, 'SDXL'):
            raise RuntimeError('ComfyUI SDXL model class is unavailable')
        if not callable(getattr(_t4_sd, 'load_checkpoint_guess_config', None)):
            raise RuntimeError('ComfyUI checkpoint loader is unavailable')

        # Load only the isolated binary named by the matching manifest. An
        # unrelated module with the same import name is never used accidentally.
        _t4_native = _t4_sys.modules.get(_t4_module_name)
        if _t4_native is None:
            _t4_spec = _t4_importlib.spec_from_file_location(
                _t4_module_name, str(_t4_binary))
            if _t4_spec is None or _t4_spec.loader is None:
                raise RuntimeError('native extension loader is unavailable')
            _t4_native = _t4_importlib.module_from_spec(_t4_spec)
            _t4_spec.loader.exec_module(_t4_native)
            _t4_sys.modules[_t4_module_name] = _t4_native
        if (getattr(_t4_native, '__phone_studio_commit__', None) != _t4_commit
                or getattr(_t4_native, '__phone_studio_patch__', None) != _t4_patch
                or _T4_Path(getattr(_t4_native, '__file__', '')).resolve()
                != _t4_binary.resolve()
                or not callable(getattr(_t4_native, 'fwd', None))):
            raise RuntimeError('native extension identity/API does not match')

        _t4_capabilities = {0: (7, 5)}
        _t4_first_use = False
        _t4_is_compiling = getattr(
            getattr(_t4_torch, 'compiler', None), 'is_compiling', None)

        def _ps_t4_attention_override(
                default_attention, q, k, v, heads, mask=None,
                attn_precision=None, skip_reshape=False,
                skip_output_reshape=False, **kwargs):
            nonlocal _t4_first_use

            def _t4_fallback():
                return default_attention(
                    q, k, v, heads, mask=mask, attn_precision=attn_precision,
                    skip_reshape=skip_reshape,
                    skip_output_reshape=skip_output_reshape, **kwargs)

            # The phone can disable per-image extras while keeping the native
            # build/model warm. Resolve the executor-scoped policy at each call.
            _t4_speed_policy = globals().get('_phone_studio_speed_extras_enabled')
            if callable(_t4_speed_policy) and not _t4_speed_policy():
                return _t4_fallback()

            # Only the known PyTorch path is replaced. Future preferred kernels,
            # masks, custom options, grouped heads and backward/tracing calls
            # retain the original implementation and all of its arguments.
            if (default_attention is not _t4_default_raw or mask is not None
                    or type(heads) is not int or not 1 <= heads <= 64
                    or type(skip_reshape) is not bool
                    or type(skip_output_reshape) is not bool
                    or _t4_torch.is_grad_enabled()
                    or _t4_torch.jit.is_tracing() or _t4_torch.jit.is_scripting()
                    or (callable(_t4_is_compiling) and _t4_is_compiling())
                    or any(type(t) is not _t4_torch.Tensor for t in (q, k, v))):
                return _t4_fallback()
            _t4_known_keys = {'transformer_options', '_inside_attn_wrapper',
                              'scale', 'enable_gqa'}
            if (any(key not in _t4_known_keys for key in kwargs)
                    or ('enable_gqa' in kwargs
                        and type(kwargs['enable_gqa']) is not bool)
                    or kwargs.get('enable_gqa', False)):
                return _t4_fallback()
            if (_t4_attention.get_attn_precision(attn_precision, q.dtype)
                    not in (None, _t4_torch.float16)):
                return _t4_fallback()
            if any(t.device != q.device or not t.is_cuda
                   or t.dtype != _t4_torch.float16 or t.requires_grad
                   or t.layout != _t4_torch.strided for t in (q, k, v)):
                return _t4_fallback()
            _t4_device = q.device.index
            if _t4_device not in _t4_capabilities:
                _t4_capabilities[_t4_device] = (
                    _t4_torch.cuda.get_device_capability(q.device))
            if _t4_capabilities[_t4_device] != (7, 5):
                return _t4_fallback()

            if skip_reshape:
                # ComfyUI skip_reshape inputs are B,H,L,D, not B,L,H,D.
                if (any(t.ndim != 4 for t in (q, k, v))
                        or any(t.shape[0] != q.shape[0]
                               or t.shape[1] != heads or t.shape[3] != 64
                               for t in (q, k, v))
                        or k.shape[2] != v.shape[2]):
                    return _t4_fallback()
                _t4_batch, _, _t4_q_len, _t4_dim = q.shape
                _t4_k_len = k.shape[2]
            else:
                if (any(t.ndim != 3 for t in (q, k, v))
                        or any(t.shape[0] != q.shape[0]
                               or t.shape[2] != heads * 64 for t in (q, k, v))
                        or k.shape[1] != v.shape[1]):
                    return _t4_fallback()
                _t4_batch, _t4_q_len, _ = q.shape
                _t4_dim, _t4_k_len = 64, k.shape[1]
            # Bound this experimental path to ordinary SDXL workloads. Larger
            # batches/sequences remain eligible for the stock memory policy.
            if (not 1 <= _t4_batch <= 4 or not 1 <= _t4_q_len <= 16384
                    or not 1 <= _t4_k_len <= 16384
                    or sum(t.numel() * t.element_size() for t in (q, k, v))
                    > 128 * 1024 * 1024):
                return _t4_fallback()
            _t4_scale = kwargs.get('scale', None)
            if _t4_scale is None:
                _t4_scale = _t4_dim ** -0.5
            elif (type(_t4_scale) not in (int, float)
                    or not _t4_math.isfinite(_t4_scale)):
                return _t4_fallback()
            # The native binding takes a positive float32 scale. Do not send
            # a Python double that would underflow to zero or overflow there.
            if not 1.1754943508222875e-38 <= _t4_scale <= 3.4028234663852886e38:
                return _t4_fallback()

            # These transformations never cast Q/K/V. Full contiguous storage
            # is mandatory: checking stride(-1) alone is unsafe for this kernel.
            with _t4_torch.cuda.device(q.device):
                if skip_reshape:
                    _t4_q, _t4_k, _t4_v = (
                        t.transpose(1, 2).contiguous() for t in (q, k, v))
                else:
                    _t4_q, _t4_k, _t4_v = (
                        t.reshape(_t4_batch, t.shape[1], heads, _t4_dim).contiguous()
                        for t in (q, k, v))
                # A contiguous narrow view can still have an unaligned storage
                # offset; the kernel assumes 16-byte vector-load alignment.
                if any(t.data_ptr() % 16 for t in (_t4_q, _t4_k, _t4_v)):
                    return _t4_fallback()
                # The pinned installer patches the CUDA launch to PyTorch's
                # current stream. Do not catch a failed native CUDA launch and
                # retry SDPA on a potentially poisoned CUDA context.
                _t4_out, _t4_lse = _t4_native.fwd(
                    _t4_q, _t4_k, _t4_v, float(_t4_scale), False)
                del _t4_lse
                if skip_output_reshape:
                    _t4_result = _t4_out.transpose(1, 2)
                else:
                    _t4_result = _t4_out.reshape(
                        _t4_batch, _t4_q_len, heads * _t4_dim)
            if not _t4_first_use:
                _t4_first_use = True
                _t4_log.info('[phone-studio] T4 FlashAttention used for eligible '
                             'SDXL FP16 head-64 attention; other calls use PyTorch.')
            return _t4_result

        _t4_original_loader = _t4_sd.load_checkpoint_guess_config

        @_t4_functools.wraps(_t4_original_loader)
        def _ps_load_checkpoint_with_t4_attention(*args, **kwargs):
            _t4_loaded = _t4_original_loader(*args, **kwargs)
            if (not isinstance(_t4_loaded, (tuple, list)) or not _t4_loaded
                    or _t4_loaded[0] is None):
                return _t4_loaded
            _t4_patcher = _t4_loaded[0]
            # Inspect the detected class instead of assuming a model family
            # from its filename. CLIP/VAE and non-SDXL models are untouched.
            if type(getattr(_t4_patcher, 'model', None)) is not _t4_model_base.SDXL:
                return _t4_loaded
            try:
                _t4_options = _t4_patcher.model_options
                if not isinstance(_t4_options, dict):
                    return _t4_loaded
                _t4_transformer = _t4_options.get('transformer_options', {})
                if (not isinstance(_t4_transformer, dict)
                        or 'optimized_attention_override' in _t4_transformer):
                    return _t4_loaded
                # Do not displace a checkpoint's preferred/quantized attention.
                if any(isinstance(module, _t4_attention.ComfyAttention)
                       and module.function is not None
                       for module in _t4_patcher.model.diffusion_model.modules()):
                    return _t4_loaded
                _t4_clone = _t4_patcher.clone()
                _t4_clone.model_options = dict(_t4_clone.model_options)
                _t4_clone.model_options['transformer_options'] = dict(
                    _t4_clone.model_options.get('transformer_options', {}))
                _t4_clone.model_options['transformer_options'][
                    'optimized_attention_override'] = _ps_t4_attention_override
                _t4_result = [_t4_clone, *_t4_loaded[1:]]
                return tuple(_t4_result) if isinstance(_t4_loaded, tuple) else _t4_result
            except (AttributeError, TypeError, KeyError) as _t4_exc:
                _t4_log.warning('[phone-studio] T4 attention model hook skipped: %s',
                                _t4_exc)
                return _t4_loaded

        _t4_sd.load_checkpoint_guess_config = _ps_load_checkpoint_with_t4_attention
        _t4_log.info('[phone-studio] Optional T4 FlashAttention hook enabled; '
                     'no speed or image-equivalence result has been measured.')
    except Exception as _t4_exc:
        # Setup failure occurs before any custom kernel is dispatched. Existing
        # PyTorch attention remains installed; no partial model hook is left.
        _t4_log.warning('[phone-studio] Optional T4 FlashAttention unavailable '
                        '(%s); using the existing PyTorch path.', _t4_exc)


_ps_install_t4_flash_attention()
# --- End optional T4 Fast attention bridge hook ---

# --- Optional bounded conditioning cache ---
"""Optional render-local SDXL text-conditioning cache for BRIDGE_CODE.

Embed after the existing checkpoint-load wrappers and T4 attention snippet,
before the bridge's route/node registrations. Inactive unless
PHONE_STUDIO_FAST_CONDITIONING_CACHE=1. No dependency is installed here.

This caches upstream CONDCrossAttn.concat plus its usual FP16 CUDA transfer;
attention projections, sampling arithmetic and global COND classes are untouched.
Current primary-source contracts:
https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/conds.py
https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/samplers.py
https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/sampler_helpers.py
https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/model_base.py
https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/patcher_extension.py
"""


def _ps_install_conditioning_cache():
    import functools
    import inspect
    import logging
    import math
    import os
    import types

    if os.environ.get('PHONE_STUDIO_FAST_CONDITIONING_CACHE', '0') != '1':
        return
    log = logging.getLogger('phone-studio.conditioning-cache')
    marker = '_phone_studio_conditioning_cache_v1'
    wrapper_key = 'phone_studio_conditioning_cache_v1'
    budget = 8 * 1024 * 1024  # Private CPU snapshots plus retained GPU tensors.
    margin = 64 * 1024 * 1024
    skipped = set()

    def skip(reason):
        if reason not in skipped and len(skipped) < 16:
            skipped.add(reason)
            log.info('[phone-studio] Conditioning cache skipped: %s; stock path retained.', reason)

    try:
        import torch
        import comfy.conds as conds_module
        import comfy.model_base as model_base
        import comfy.model_management as management
        import comfy.model_patcher as patchers
        import comfy.patcher_extension as extension
        import comfy.sampler_helpers as helpers
        import comfy.samplers as samplers
        import comfy.sd as sd
        import comfy.supported_models as supported_models
        import latent_preview
        from comfy.ldm.modules import attention
        from comfy.ldm.modules.diffusionmodules.openaimodel import UNetModel

        if getattr(sd.load_checkpoint_guess_config, marker, False):
            return
        if (os.environ.get('PHONE_STUDIO_FAST_DENOISE_CACHE', '0') == '1'
                or os.environ.get('PHONE_STUDIO_T4_FLASH_ATTENTION', '0') == '1'):
            raise RuntimeError('turn EasyCache and T4 FlashAttention off for a separate comparison')
        if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
            raise RuntimeError('a CUDA T4/Turing SM7.5 runtime is required')
        is_compiling = getattr(getattr(torch, 'compiler', None), 'is_compiling', None)
        if not callable(is_compiling):
            raise RuntimeError('PyTorch compile-state guard is unavailable')
        cross_class = conds_module.CONDCrossAttn
        original_concat = cross_class.concat
        sample_method = inspect.unwrap(samplers.KSAMPLER.sample)
        expected = ('self', 'model_wrap', 'sigmas', 'extra_args', 'callback',
                    'noise', 'latent_image', 'denoise_mask', 'disable_pbar')
        if tuple(inspect.signature(sample_method).parameters) != expected:
            raise RuntimeError('ComfyUI sampler signature changed')
        if (tuple(inspect.signature(original_concat).parameters) != ('self', 'others')
                or cross_class.process_cond is not conds_module.CONDRegular.process_cond
                or cross_class._copy_with is not conds_module.CONDRegular._copy_with
                or tuple(inspect.signature(patchers.ModelPatcher.add_wrapper_with_key).parameters)
                != ('self', 'wrapper_type', 'key', 'wrapper')):
            raise RuntimeError('ComfyUI conditioning/model wrapper API changed')
        inner_source = inspect.getsource(inspect.unwrap(samplers.CFGGuider.inner_sample))
        if ('process_conds(' not in inner_source
                or 'WrappersMP.SAMPLER_SAMPLE' not in inner_source
                or inner_source.index('process_conds(') > inner_source.index('WrappersMP.SAMPLER_SAMPLE')
                or 'model.wrappers' not in inspect.getsource(helpers.prepare_model_patcher)
                or 'cast_to_device(context, device, dtype)' not in
                inspect.getsource(inspect.unwrap(model_base.BaseModel._apply_model))):
            raise RuntimeError('ComfyUI render/cast hook contract changed')
        sampler_function = samplers.k_diffusion_sampling.sample_dpmpp_2m
        callback_codes = tuple(
            code for code in inspect.unwrap(latent_preview.prepare_callback).__code__.co_consts
            if isinstance(code, types.CodeType) and code.co_name == 'callback')
        if not callback_codes:
            raise RuntimeError('stock preview callback contract changed')
        patcher_types = (patchers.ModelPatcher, patchers.CoreModelPatcher)

        def empty(value):
            if isinstance(value, dict):
                return all(empty(item) for item in value.values())
            if isinstance(value, (list, tuple)):
                return not value
            return value is None

        def own_wrappers(registry):
            if not isinstance(registry, dict):
                return False
            for stage, groups in registry.items():
                if not isinstance(groups, dict):
                    return False
                for key, functions in groups.items():
                    if not isinstance(functions, list):
                        return False
                    if functions and (stage != extension.WrappersMP.SAMPLER_SAMPLE
                                      or key != wrapper_key
                                      or any(fn is not sample_with_cache for fn in functions)):
                        return False
            return True

        def plain_options(options):
            if not isinstance(options, dict) or set(options) - {'transformer_options', 'to_load_options'}:
                return False
            transformer = options.get('transformer_options', {})
            load_options = options.get('to_load_options', {})
            if (not isinstance(transformer, dict) or not isinstance(load_options, dict)
                    or set(transformer) - {'wrappers', 'callbacks', 'sample_sigmas'}
                    or set(load_options) - {'wrappers', 'callbacks'}):
                return False
            return all(own_wrappers(value.get('wrappers', {}))
                       and empty(value.get('callbacks', {}))
                       for value in (transformer, load_options))

        def model_reason(patcher):
            if type(patcher) not in patcher_types:
                return 'custom model patcher'
            model = patcher.model
            if (type(model) is not model_base.SDXL
                    or type(model.model_config) is not supported_models.SDXL
                    or type(model.diffusion_model) is not UNetModel
                    or model.model_config.custom_operations is not None
                    or model.model_config.quant_config is not None):
                return 'only stock SDXL base models are supported'
            if (model.get_dtype_inference() != torch.float16
                    or model.diffusion_model.training):
                return 'only evaluation-mode FP16 inference is supported'
            if (not plain_options(patcher.model_options)
                    or not own_wrappers(patcher.wrappers)
                    or not empty(patcher.callbacks)
                    or any(getattr(patcher, key, None) for key in
                           ('object_patches', 'weight_wrapper_patches', 'injections',
                            'additional_models', 'hook_patches', 'forced_hooks', 'current_hooks'))):
                return 'custom model, sampler, attention, or conditioning hooks'
            attachments = patcher.attachments
            if (not isinstance(attachments, dict) or set(attachments) - {'lora_metadata'}
                    or ('lora_metadata' in attachments
                        and not isinstance(attachments['lora_metadata'], dict))):
                return 'custom model attachments'
            for module in model.diffusion_model.modules():
                if (any(getattr(module, key, None) for key in
                        ('_forward_hooks', '_forward_pre_hooks', '_backward_hooks',
                         '_torchdynamo_orig_callable'))
                        or 'forward' in module.__dict__ or '_forward' in module.__dict__
                        or (isinstance(module, attention.ComfyAttention)
                            and module.function is not None)):
                    return 'custom module hooks or preferred attention'
            return None

        def checked_model_reason(patcher):
            try:
                return model_reason(patcher)
            except (AttributeError, TypeError, KeyError):
                return 'ComfyUI model hook API changed'

        def sample_with_cache(executor, model_wrap, sigmas, extra_args, callback,
                              noise, latent_image=None, denoise_mask=None, disable_pbar=False):
            def stock(reason):
                skip(reason)
                return executor(model_wrap, sigmas, extra_args, callback, noise,
                                latent_image, denoise_mask, disable_pbar)

            speed_extras = globals().get('_phone_studio_speed_extras_enabled')
            if callable(speed_extras) and not speed_extras():
                return stock('speed extras are off for this phone request')
            sampler = executor.class_obj
            if (type(sampler) is not samplers.KSAMPLER
                    or inspect.unwrap(getattr(executor.original, '__func__', executor.original))
                    is not sample_method
                    or sampler.sampler_function is not sampler_function
                    or sampler.extra_options or sampler.inpaint_options
                    or type(model_wrap) is not samplers.CFGGuider
                    or model_wrap.cfg != 6.5
                    or torch.is_grad_enabled() or is_compiling()
                    or torch.jit.is_tracing() or torch.jit.is_scripting()):
                return stock('stock CFG 6.5 DPM++ 2M inference is required')
            if (denoise_mask is not None or type(noise) is not torch.Tensor
                    or noise.ndim != 4 or noise.shape[0] != 1 or noise.shape[1] != 4
                    or not noise.is_cuda or noise.requires_grad
                    or type(sigmas) is not torch.Tensor or sigmas.ndim != 1
                    or sigmas.numel() not in (19, 25)
                    or sigmas.device != noise.device
                    or (callback is not None and (type(callback) is not types.FunctionType
                                                 or callback.__code__ not in callback_codes))):
                return stock('batch-one two-preset sampling with the stock progress callback is required')
            if (not isinstance(extra_args, dict)
                    or set(extra_args) - {'model_options', 'seed'}
                    or not plain_options(extra_args.get('model_options', {}))
                    or not plain_options(model_wrap.model_options)):
                return stock('custom runtime sampling options')
            reason = checked_model_reason(model_wrap.model_patcher)
            if reason:
                return stock(reason)
            if (model_wrap.inner_model is not model_wrap.model_patcher.model
                    or model_wrap.inner_model.current_patcher is not model_wrap.model_patcher
                    or model_wrap.model_patcher.load_device != noise.device):
                return stock('model/device identity changed')
            device = noise.device
            with torch.cuda.device(device):
                if torch.cuda.get_device_capability(device) != (7, 5):
                    return stock('only T4/Turing SM7.5 is supported')
                if torch.cuda.mem_get_info(device)[0] < budget + margin:
                    return stock('insufficient free CUDA memory for the bounded cache')
                stream_id = torch.cuda.current_stream(device).cuda_stream
            original_conds = model_wrap.conds
            if not isinstance(original_conds, dict) or set(original_conds) != {'positive', 'negative'}:
                return stock('one plain positive and negative condition are required')
            source_tensors = []
            allowed_keys = {'cross_attn', 'pooled_output', 'model_conds', 'uuid'}
            for name in ('positive', 'negative'):
                entries = original_conds[name]
                if (not isinstance(entries, list) or len(entries) != 1
                        or not isinstance(entries[0], dict) or set(entries[0]) - allowed_keys):
                    return stock('scheduled, masked, regional, or custom conditioning')
                model_conds = entries[0].get('model_conds')
                if (not isinstance(model_conds, dict) or set(model_conds) != {'c_crossattn', 'y'}
                        or type(model_conds['c_crossattn']) is not cross_class
                        or type(model_conds['y']) is not conds_module.CONDRegular):
                    return stock('stock SDXL cross-attention and ADM conditioning are required')
                tensor = model_conds['c_crossattn'].cond
                if (type(tensor) is not torch.Tensor or tensor.ndim != 3
                        or tuple(tensor.shape)[::2] != (1, 2048)
                        or not 1 <= tensor.shape[1] <= 308
                        or tensor.device.type != 'cpu' or tensor.requires_grad
                        or tensor.layout != torch.strided
                        or tensor.dtype not in (torch.float16, torch.float32)):
                    return stock('bounded CPU [1,L,2048] text embeddings are required')
                source_tensors.append(tensor)
            snapshot_bytes = sum(t.numel() * t.element_size() for t in source_tensors)
            maximum_length = math.lcm(*(t.shape[1] for t in source_tensors))
            first_output_bytes = 2 * maximum_length * 2048 * 2
            if snapshot_bytes + first_output_bytes > budget:
                return stock('conditioning exceeds the 8 MiB total budget')

            # Copies isolate this render from node-output caching and lack of
            # version counters on inference tensors. Stock SDXL only reads them.
            with torch.inference_mode(False), torch.no_grad():
                snapshots = [tensor.detach().clone() for tensor in source_tensors]
            source_ids = {id(tensor) for tensor in snapshots}
            cache = {}
            retained_bytes = snapshot_bytes
            hits = misses = 0

            class RenderCrossAttn(cross_class):
                def concat(self, others):
                    nonlocal retained_bytes, hits, misses
                    inputs = [self, *others]
                    tensors = [item.cond for item in inputs]
                    # Stock process_cond returns the same source for batch one.
                    # An unexpected repeat/view/condition keeps ordinary concat.
                    if (len(inputs) not in (1, 2)
                            or any(type(item) is not RenderCrossAttn for item in inputs)
                            or any(id(tensor) not in source_ids for tensor in tensors)
                            or torch.cuda.current_stream(device).cuda_stream != stream_id):
                        skip('conditioning batch identity or CUDA stream changed')
                        return original_concat(self, others)
                    key = (tuple((id(tensor), tuple(tensor.shape), tuple(tensor.stride()), tensor.dtype)
                                 for tensor in tensors), device, torch.float16, stream_id)
                    if key in cache:
                        hits += 1
                        return cache[key]
                    output_length = math.lcm(*(tensor.shape[1] for tensor in tensors))
                    output_bytes = len(tensors) * output_length * 2048 * 2
                    if len(cache) >= 4 or retained_bytes + output_bytes > budget:
                        skip('render-local cache layout/byte cap reached')
                        return original_concat(self, others)
                    # Preserve upstream CPU concatenation/padding and the same
                    # cast operation used by BaseModel. No early rounding.
                    combined = original_concat(self, others)
                    with torch.cuda.device(device):
                        output = management.cast_to_device(combined, device, torch.float16)
                    cache[key] = output
                    retained_bytes += output_bytes
                    misses += 1
                    return output

            render_conds = dict(original_conds)
            for name, snapshot in zip(('positive', 'negative'), snapshots):
                entry = dict(original_conds[name][0])
                entry['model_conds'] = dict(entry['model_conds'])
                entry['model_conds']['c_crossattn'] = RenderCrossAttn(snapshot)
                render_conds[name] = [entry]
            del entry, snapshot
            model_wrap.conds = render_conds
            try:
                # Numerical/CUDA failures propagate; never retry a failed GPU
                # call on another path. No cache survives an interrupted render.
                return executor(model_wrap, sigmas, extra_args, callback, noise,
                                latent_image, denoise_mask, disable_pbar)
            finally:
                model_wrap.conds = original_conds
                cache.clear()
                snapshots.clear()
                render_conds.clear()
                log.info('[phone-studio] Conditioning cache released: hits=%d misses=%d retained_bytes=%d cap_bytes=%d.',
                         hits, misses, retained_bytes, budget)

        original_loader = sd.load_checkpoint_guess_config

        @functools.wraps(original_loader)
        def load_with_cache(*args, **kwargs):
            loaded = original_loader(*args, **kwargs)
            if not isinstance(loaded, (tuple, list)) or not loaded or loaded[0] is None:
                return loaded
            patcher = loaded[0]
            reason = checked_model_reason(patcher)
            if reason:
                skip(reason)
                return loaded
            # Ordinary ModelPatcher clones share weights and preserve wrappers;
            # adding ours does not force full residency or modify CLIP/VAE.
            try:
                clone = patcher.clone()
                if not any(fn is sample_with_cache for fn in
                           clone.get_wrappers(extension.WrappersMP.SAMPLER_SAMPLE, wrapper_key)):
                    clone.add_wrapper_with_key(extension.WrappersMP.SAMPLER_SAMPLE,
                                               wrapper_key, sample_with_cache)
            except (AttributeError, TypeError, KeyError):
                skip('ComfyUI model clone/wrapper API changed')
                return loaded
            result = [clone, *loaded[1:]]
            return tuple(result) if isinstance(loaded, tuple) else result

        setattr(load_with_cache, marker, True)
        sd.load_checkpoint_guess_config = load_with_cache
        log.info('[phone-studio] Optional conditioning cache hook enabled (8 MiB, render-local); no speed or image-equivalence result measured.')
    except Exception as error:
        # Initialization/API checks occur before attaching the loader hook and
        # before any optional tensor transfer. Unsupported installs stay stock.
        log.warning('[phone-studio] Conditioning cache unavailable (%s); stock path retained.', error)


_ps_install_conditioning_cache()
# --- End optional bounded conditioning cache ---

# Fill only missing checkpoint selections. Explicit phone/UI choices win.
def _phone_studio_default_checkpoint(json_data):
    global _STARTUP_WARMUP_REUSE_ALLOWED, _COMPLETED_WARMUP
    # Reuse only before subsequent work could change model residency.
    # Invalid requests also disable it conservatively; no real work is skipped.
    if json_data.get('client_id') != 'phone-studio-startup-warmup':
        _STARTUP_WARMUP_REUSE_ALLOWED = False
        _COMPLETED_WARMUP = None
    prompt = json_data.get('prompt')
    if DEFAULT_CHECKPOINT and isinstance(prompt, dict):
        for node in prompt.values():
            if not isinstance(node, dict) or node.get('class_type') != 'CheckpointLoaderSimple':
                continue
            inputs = node.get('inputs')
            if inputs is None:
                inputs = node['inputs'] = {}
            if not isinstance(inputs, dict):
                continue
            selected = inputs.get('ckpt_name')
            if selected is None or (isinstance(selected, str) and not selected.strip()):
                inputs['ckpt_name'] = DEFAULT_CHECKPOINT
    return json_data

PromptServer.instance.add_on_prompt_handler(_phone_studio_default_checkpoint)

# --- Optional Fast denoising cache; remove this block to undo the hooks ---
# Approximate denoising changes pixels. Requested ON; no model/step/CFG/
# sampler/resolution changes. Uses ComfyUI's built-in experimental EasyCache.
# Upstream schema: comfy_extras/nodes_easycache.py, EasyCacheNode.
_PS_FAST_DENOISE_CACHE = os.environ.get('PHONE_STUDIO_FAST_DENOISE_CACHE', '0') == '1'
_PS_EASYCACHE_INPUTS = {
    'reuse_threshold': 0.2,
    'start_percent': 0.15,
    'end_percent': 0.95,
    'verbose': False,
}

def _phone_studio_easycache_graph(body):
    if not _PS_FAST_DENOISE_CACHE or not isinstance(body, dict):
        return None
    extra = body.get('extra_data')
    if not isinstance(extra, dict) or not extra.get('phone_studio_request_id'):
        return None
    if extra.get('phone_studio_warmup') or extra.get('phone_studio_speed_extras', True) is False:
        return None
    graph = body.get('prompt')
    if _ps_is_edit_graph(graph):
        return None  # Masked/patched edit combinations are deliberately not approximated.
    return graph if isinstance(graph, dict) else None

def _phone_studio_easycache_targets(graph):
    targets = []
    for node_id, node in graph.items():
        if not isinstance(node, dict) or node.get('class_type') != 'KSampler':
            continue
        inputs = node.get('inputs')
        if not isinstance(inputs, dict):
            continue
        link = inputs.get('model')
        if not isinstance(link, list) or len(link) != 2 or not isinstance(link[0], str):
            continue  # Preserve malformed graphs for standard /prompt validation.
        source = graph.get(link[0])
        if not isinstance(source, dict):
            continue
        if source.get('class_type') == 'EasyCache':
            continue  # Never stack a second cache over an existing wrapper.
        targets.append((node_id, link))
    return targets

def _phone_studio_add_easycache(body):
    graph = _phone_studio_easycache_graph(body)
    if graph is None:
        return body
    targets = _phone_studio_easycache_targets(graph)
    if not targets:
        return body
    # Build a replacement graph before publishing it: on_prompt exceptions are
    # swallowed by ComfyUI, so never leave the input graph partially rewritten.
    patched = dict(graph)
    for node_id, model_link in targets:
        cache_inputs = dict(_PS_EASYCACHE_INPUTS, model=list(model_link))
        wrapper_id = next((key for key, value in patched.items()
                           if isinstance(value, dict)
                           and value.get('class_type') == 'EasyCache'
                           and value.get('inputs') == cache_inputs), None)
        if wrapper_id is None:
            index = 1
            wrapper_id = 'phone_studio_easycache_' + str(index)
            while wrapper_id in patched:
                index += 1
                wrapper_id = 'phone_studio_easycache_' + str(index)
            patched[wrapper_id] = {
                'class_type': 'EasyCache',
                'inputs': cache_inputs,
                '_meta': {'title': 'Phone Studio Fast denoise cache (approximate)'},
            }
        sampler = dict(graph[node_id])
        sampler['inputs'] = dict(sampler['inputs'], model=[wrapper_id, 0])
        patched[node_id] = sampler
    body['prompt'] = patched  # Hidden PROMPT/PNG metadata includes the cache node.
    print(f'[phone-studio] Fast denoise cache applied to {len(targets)} sampler(s); approximate output')
    return body

if _PS_FAST_DENOISE_CACHE:
    @web.middleware
    async def _phone_studio_easycache_guard(request, handler):
        if request.method == 'POST' and request.path == '/prompt':
            try:
                body = await request.json()
            except Exception:
                body = None  # Let ComfyUI report malformed JSON normally.
            graph = _phone_studio_easycache_graph(body)
            if graph is not None and _phone_studio_easycache_targets(graph):
                # Built-in nodes load after custom-node import on some versions.
                # Check the final registry at request time, never during import.
                import nodes as _easycache_nodes
                if 'EasyCache' not in _easycache_nodes.NODE_CLASS_MAPPINGS:
                    message = ('Fast cache requires EasyCache. Update ComfyUI or set '
                               'FAST_DENOISE_CACHE = False and restart the server.')
                    print('[phone-studio] Fast denoise cache unavailable; rejecting app prompt')
                    return web.json_response({
                        'error': {'type': 'phone_studio_easycache_unavailable',
                                  'message': message, 'details': message, 'extra_info': {}},
                        'node_errors': {},
                    }, status=400)
        return await handler(request)

    # request.json() caches raw bytes; the normal /prompt handler can reread it.
    # Graph changes belong in on_prompt, avoiding private request-body overrides.
    PromptServer.instance.app.middlewares.append(_phone_studio_easycache_guard)
    PromptServer.instance.add_on_prompt_handler(_phone_studio_add_easycache)
    print('[phone-studio] Fast denoise cache ON: app requests only; warmups excluded; output may change')
else:
    print('[phone-studio] Fast denoise cache disabled by notebook setting')


def require_token(request):
    supplied = request.headers.get('X-Phone-Studio-Token', '')
    if not PHONE_TOKEN or not hmac.compare_digest(supplied, PHONE_TOKEN):
        raise web.HTTPUnauthorized(text='Phone Studio token required')

def safe_filename(value):
    value = Path(value).name
    value = re.sub(r'[^A-Za-z0-9._ -]+', '_', value).strip(' .')
    if not value.lower().endswith('.safetensors'):
        raise ValueError('Only SafeTensors model files are supported')
    if not value or value == '.safetensors':
        raise ValueError('Model file name is invalid')
    return value

def models_in(folder):
    path = MODELS_ROOT / folder
    if not path.exists():
        return []
    return sorted(str(item.relative_to(path)).replace('\\', '/') for item in path.rglob('*') if item.suffix.lower() in {'.safetensors', '.ckpt', '.pt'} and item.is_file())

# Only generation preflight opts into a recent catalog. Connect/Refresh and
# older clients still receive a fresh scan. Work runs off the aiohttp event loop
# so slow Drive metadata cannot block health, history, or progress delivery.
CATALOG_CACHE_SECONDS = max(0.0, float(os.environ.get('PHONE_STUDIO_CATALOG_CACHE_SECONDS', '300')))
_CATALOG_LOCK = threading.Lock()
_CATALOG = None
_CATALOG_UPDATED = 0.0

def model_catalog(allow_cached=False):
    global _CATALOG, _CATALOG_UPDATED
    with _CATALOG_LOCK:
        if (allow_cached and _CATALOG is not None
                and time.monotonic() - _CATALOG_UPDATED < CATALOG_CACHE_SECONDS):
            return _CATALOG
        checkpoints = models_in('checkpoints')
        if DEFAULT_CHECKPOINT in checkpoints:
            checkpoints.remove(DEFAULT_CHECKPOINT)
            checkpoints.insert(0, DEFAULT_CHECKPOINT)
        _CATALOG = {
            'checkpoints': checkpoints, 'loras': models_in('loras'),
            'defaultCheckpoint': DEFAULT_CHECKPOINT,
        }
        _CATALOG_UPDATED = time.monotonic()
        return _CATALOG

def invalidate_model_catalog():
    global _CATALOG
    with _CATALOG_LOCK:
        _CATALOG = None

def _completed_startup_warmup():
    # The launcher publishes this only after successful history + preview.
    # A recycled/restarted process has a new INSTANCE_ID; stale proof is ignored.
    global _COMPLETED_WARMUP
    if not _STARTUP_WARMUP_REUSE_ALLOWED:
        return None
    if _COMPLETED_WARMUP is not None:
        return _COMPLETED_WARMUP
    try:
        import json
        with COMPLETED_WARMUP_FILE.open(encoding='utf-8') as handle:
            raw = handle.read(4097)
        if len(raw) > 4096:
            return None
        record = json.loads(raw)
        if not isinstance(record, dict) or record.get('instanceId') != INSTANCE_ID:
            return None
        warmup = record.get('completedWarmup')
        if not isinstance(warmup, dict) or warmup.get('guidanceMode') != 'cfg':
            return None
        for key in ('model', 'sampler', 'scheduler', 'promptId'):
            if not isinstance(warmup.get(key), str) or not warmup[key].strip():
                return None
        for key in ('width', 'height'):
            if type(warmup.get(key)) is not int or warmup[key] <= 0 or warmup[key] % 8:
                return None
        _COMPLETED_WARMUP = {key: warmup[key] for key in
                             ('model', 'width', 'height', 'sampler', 'scheduler', 'guidanceMode', 'promptId')}
        return _COMPLETED_WARMUP
    except (OSError, ValueError, TypeError):
        return None  # Optional optimization: retain the phone's existing warm-up.


def _phone_studio_edit_capabilities():
    import nodes as _edit_nodes
    registered = set(_edit_nodes.NODE_CLASS_MAPPINGS)
    required = {'CheckpointLoaderSimple', 'CLIPTextEncode', 'LoadImage', 'LoadImageMask',
                'VAEEncode', 'VAEEncodeForInpaint', 'SetLatentNoiseMask', 'KSampler',
                'VAEDecode', 'ImageCompositeMasked', 'SaveImage'}
    editing = required <= registered and _PS_SPEED_SWITCH_AVAILABLE
    fooocus_nodes = {'INPAINT_LoadFooocusInpaint', 'INPAINT_ApplyFooocusInpaint',
                     'INPAINT_VAEEncodeInpaintConditioning'}
    return {'editing': editing,
            'fooocusInpaint': editing and os.environ.get('PHONE_STUDIO_FOOOCUS_READY') == '1'
                and fooocus_nodes <= registered,
            'editSpeedExtras': False,
            'fooocusSourceRevision': os.environ.get('PHONE_STUDIO_FOOOCUS_SOURCE_REVISION', '')}


# --- Authenticated worker baseline diagnostics ---
def _phone_studio_worker_diagnostics():
    import hashlib
    import json
    memory = {}
    try:
        for line in Path('/proc/meminfo').read_text(encoding='ascii').splitlines():
            key, _, value = line.partition(':')
            if key in {'MemAvailable', 'SwapFree', 'SwapTotal'}:
                memory[key + 'Bytes'] = int(value.split()[0]) * 1024
        for line in Path('/proc/self/status').read_text(encoding='ascii').splitlines():
            key, _, value = line.partition(':')
            if key in {'VmRSS', 'VmHWM', 'VmSwap'}:
                memory[key + 'Bytes'] = int(value.split()[0]) * 1024
    except (OSError, ValueError):
        pass
    try:
        free, total = torch.cuda.mem_get_info()
        memory.update(gpuFreeBytes=free, gpuTotalBytes=total)
    except Exception:
        pass
    try:
        bridge_hash = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    except OSError:
        bridge_hash = ''
    try:
        effective = json.loads(os.environ.get('PHONE_STUDIO_EFFECTIVE_OPTIONS', '{}'))
    except ValueError:
        effective = {}
    return {'runtimeId': os.environ.get('PHONE_STUDIO_RUNTIME_ID', ''),
            'buildFingerprint': os.environ.get('PHONE_STUDIO_WORKER_BUILD', ''),
            'bridgeSha256': bridge_hash,
            'comfyRevision': os.environ.get('PHONE_STUDIO_COMFY_REVISION', ''),
            'torch': str(torch.__version__), 'cuda': torch.version.cuda,
            'compatibilityMode': _PS_COMPATIBILITY_MODE,
            'comfyCompilerDisabled': _PS_COMPATIBILITY_MODE,
            'attention': os.environ.get('PHONE_STUDIO_ATTENTION', 'pytorch'),
            'effectiveOptions': effective, 'memory': memory,
            'stage': 'ready' if _completed_startup_warmup() is not None else 'server_started'}
# --- End authenticated worker baseline diagnostics ---

@routes.get('/phone-studio/health')
async def phone_studio_health(request):
    require_token(request)
    body = {'ok': True, 'service': 'phone-studio-bridge', 'instanceId': INSTANCE_ID,
            'speedExtrasSupported': _PS_SPEED_SWITCH_AVAILABLE,
            'capabilities': _phone_studio_edit_capabilities(),
            'workerRuntime': _phone_studio_worker_diagnostics()}
    completed = _completed_startup_warmup()
    if completed is not None:
        body['completedWarmup'] = completed
    return web.json_response(body)

@routes.get('/phone-studio/models')
async def phone_studio_models(request):
    require_token(request)
    catalog = await asyncio.to_thread(model_catalog, request.query.get('cached') == '1')
    return web.json_response(catalog)

def download_civitai(job_id, version_id, requested_kind):
    state = DOWNLOADS[job_id]
    headers = {'Authorization': f'Bearer {CIVITAI_KEY}'} if CIVITAI_KEY else {}
    try:
        state.update(status='looking up model', progress=0)
        response = requests.get(f'https://civitai.com/api/v1/model-versions/{version_id}', headers=headers, timeout=45)
        response.raise_for_status()
        info = response.json()
        files = info.get('files') or []
        safe_files = [item for item in files if str(item.get('name', '')).lower().endswith('.safetensors') and item.get('downloadUrl')]
        file_info = next((item for item in safe_files if item.get('primary')), None) or next(iter(safe_files), None)
        if not file_info:
            raise RuntimeError('Civitai did not return a downloadable SafeTensors file for this version')
        inferred_type = str((info.get('model') or {}).get('type', '')).lower()
        kind = requested_kind if requested_kind in {'checkpoint', 'lora'} else ('lora' if 'lora' in inferred_type else 'checkpoint')
        destination_dir = MODELS_ROOT / ('loras' if kind == 'lora' else 'checkpoints')
        destination_dir.mkdir(parents=True, exist_ok=True)
        filename = safe_filename(file_info.get('name') or f'civitai_{version_id}.safetensors')
        destination = destination_dir / filename
        temp = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.part')
        state.update(status='downloading', filename=filename, progress=0)
        with requests.get(file_info['downloadUrl'], headers=headers, stream=True, allow_redirects=True, timeout=90) as stream:
            stream.raise_for_status()
            total = int(stream.headers.get('content-length') or 0)
            downloaded = 0
            with open(temp, 'wb') as handle:
                for chunk in stream.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    handle.write(chunk)
                    downloaded += len(chunk)
                    state['progress'] = round(downloaded * 100 / total, 1) if total else None
        temp.replace(destination)
        invalidate_model_catalog()
        state.update(status='complete', progress=100, filename=filename, kind=kind)
    except Exception as exc:
        state.update(status='failed', error=str(exc))

@routes.post('/phone-studio/download-model')
async def phone_studio_download(request):
    require_token(request)
    payload = await request.json()
    version_id = str(payload.get('modelVersionId', '')).strip()
    if not version_id.isdigit():
        raise web.HTTPBadRequest(text='modelVersionId must be a Civitai model-version ID')
    requested_kind = str(payload.get('kind', '')).lower()
    job_id = uuid.uuid4().hex
    DOWNLOADS[job_id] = {'status': 'queued', 'progress': 0}
    asyncio.create_task(asyncio.to_thread(download_civitai, job_id, version_id, requested_kind))
    return web.json_response({'jobId': job_id})

@routes.get('/phone-studio/downloads/{job_id}')
async def phone_studio_download_status(request):
    require_token(request)
    state = DOWNLOADS.get(request.match_info['job_id'])
    if not state:
        raise web.HTTPNotFound(text='Download job not found')
    return web.json_response(state)

# --- Track + expose recent generation jobs by request ID -------------------
# ComfyUI keeps accepted prompts running without a connected phone. A bounded
# request-ID map lets the app recover an ambiguous submission without ever
# attaching to a different, newer prompt on the same backend.
import time as _time

LATEST_JOB_LIMIT = 32
LATEST_JOBS = OrderedDict()

@web.middleware
async def _phone_studio_track_prompt(request, handler):
    body_in = {}
    if request.method == 'POST' and request.path == '/prompt':
        try:
            parsed = await request.json()
            body_in = parsed if isinstance(parsed, dict) else {}
        except Exception:
            body_in = {}
    response = await handler(request)
    extra_data = body_in.get('extra_data')
    if not isinstance(extra_data, dict):
        extra_data = {}
    is_warmup = bool(extra_data.get('phone_studio_warmup'))
    if request.method == 'POST' and request.path == '/prompt' and response.status == 200 and not is_warmup:
        try:
            import json as _json
            prompt_id = _json.loads(response.text).get('prompt_id')
            request_id = str(extra_data.get('phone_studio_request_id') or '')
            if prompt_id and request_id:
                LATEST_JOBS.pop(request_id, None)
                LATEST_JOBS[request_id] = {
                    'prompt_id': prompt_id,
                    'submitted_at': _time.time(),
                    'client_id': str(body_in.get('client_id') or ''),
                    'request_id': request_id,
                }
                while len(LATEST_JOBS) > LATEST_JOB_LIMIT:
                    LATEST_JOBS.popitem(last=False)
        except Exception:
            pass
    return response
try:
    PromptServer.instance.app.middlewares.append(_phone_studio_track_prompt)
    print('[phone-studio] request-keyed latest-job tracking enabled (last 32)')
except Exception as exc:
    print(f'[phone-studio] could not install latest-job tracking ({exc}); /phone-studio/latest-job will always report none')

@routes.get('/phone-studio/latest-job')
async def phone_studio_latest_job(request):
    require_token(request)
    requested_client_id = request.query.get('clientId', '').strip()
    requested_request_id = request.query.get('requestId', '').strip()
    if not requested_client_id or not requested_request_id:
        return web.json_response({'status': 'none'})
    job = LATEST_JOBS.get(requested_request_id)
    if job is None:
        return web.json_response({'status': 'none'})
    if requested_client_id != job.get('client_id'):
        return web.json_response({'status': 'none'})
    prompt_id = job['prompt_id']
    submitted_at = job['submitted_at']
    request_id = job['request_id']
    try:
        history = PromptServer.instance.prompt_queue.get_history(prompt_id=prompt_id)
    except Exception as exc:
        return web.json_response({'status': 'unknown', 'promptId': prompt_id, 'requestId': request_id, 'submittedAt': submitted_at, 'error': str(exc)})
    entry = history.get(prompt_id)
    if entry is None:
        return web.json_response({'status': 'pending', 'promptId': prompt_id, 'requestId': request_id, 'submittedAt': submitted_at})
    images = []
    for node_output in (entry.get('outputs') or {}).values():
        images.extend(node_output.get('images') or [])
    status_info = entry.get('status') or {}
    failed = str(status_info.get('status_str', '')).lower() == 'error'
    return web.json_response({'status': 'error' if failed else 'done', 'promptId': prompt_id, 'requestId': request_id, 'submittedAt': submitted_at, 'images': images})
# --- Replay a missed result over /ws when the phone reconnects -------------
# Best-effort, and it rests on one assumption about the app: that it reuses a
# stable clientId across reconnects (the standard ComfyUI client pattern -
# generate a UUID once, keep sending it as ?clientId=... on /ws and client_id
# in the /prompt body) rather than minting a new one every time it opens a
# socket. If it does, this makes a finished-while-offline job show up the
# instant the socket reopens, without the app doing anything different.
# If it doesn't reuse a clientId, this silently does nothing extra - it
# can't identify "the same phone" on reconnect, so there's nothing to key
# the replay on. Check the Colab log line below to see what your app is
# actually doing here; that tells you in one glance whether this can work
# at all before you rely on it.
#
# What it actually sends over the socket is a synthetic version of ComfyUI's
# own live 'executed' / 'executing' messages, reconstructed from history
# after the fact. I can't test this against your specific app from here -
# confirm it works by starting a generation, backgrounding the app before
# it finishes, waiting past completion, then reopening the app with no other
# action and checking whether the image just appears.
PENDING_REPLAY = {}

async def _phone_studio_watch_job(client_id, prompt_id):
    if not client_id or not prompt_id:
        return
    for _ in range(600):  # ~10 min ceiling; real generations finish well inside this
        try:
            history = PromptServer.instance.prompt_queue.get_history(prompt_id=prompt_id)
        except Exception:
            return
        entry = history.get(prompt_id)
        if entry is not None:
            if PromptServer.instance.sockets.get(client_id) is None:
                PENDING_REPLAY[client_id] = {'prompt_id': prompt_id, 'outputs': entry.get('outputs') or {}}
                print(f'[phone-studio] {client_id} was offline when {prompt_id} finished; queued for replay on reconnect')
            return
        await asyncio.sleep(1)

async def _phone_studio_replay_when_ready(client_id):
    for _ in range(50):  # ~5s to let the handshake finish and register the socket
        ws = PromptServer.instance.sockets.get(client_id)
        if ws is not None:
            break
        await asyncio.sleep(0.1)
    else:
        return
    payload = PENDING_REPLAY.pop(client_id, None)
    if not payload:
        return
    try:
        for node_id, node_output in (payload['outputs'] or {}).items():
            await ws.send_json({'type': 'executed', 'data': {'node': node_id, 'output': node_output, 'prompt_id': payload['prompt_id']}})
        await ws.send_json({'type': 'executing', 'data': {'node': None, 'prompt_id': payload['prompt_id']}})
        print(f"[phone-studio] replayed missed result to {client_id}")
    except Exception as exc:
        print(f'[phone-studio] replay to {client_id} failed: {exc}')

@web.middleware
async def _phone_studio_reconnect_bridge(request, handler):
    if request.method == 'POST' and request.path == '/prompt':
        try:
            body_in = await request.json()
            client_id = body_in.get('client_id')
            is_warmup = bool((body_in.get('extra_data') or {}).get('phone_studio_warmup'))
        except Exception:
            client_id = None
            is_warmup = False
        response = await handler(request)
        if client_id and not is_warmup and response.status == 200:
            try:
                import json as _json
                prompt_id = _json.loads(response.text).get('prompt_id')
                if prompt_id:
                    asyncio.create_task(_phone_studio_watch_job(client_id, prompt_id))
            except Exception:
                pass
        return response
    if request.method == 'GET' and request.path == '/ws':
        client_id = request.query.get('clientId')
        print(f'[phone-studio] /ws connect, clientId={client_id!r}, pending replay={client_id in PENDING_REPLAY}')
        if client_id and client_id in PENDING_REPLAY:
            asyncio.create_task(_phone_studio_replay_when_ready(client_id))
    return await handler(request)
try:
    PromptServer.instance.app.middlewares.append(_phone_studio_reconnect_bridge)
    print('[phone-studio] reconnect-replay bridge enabled')
except Exception as exc:
    print(f'[phone-studio] could not install reconnect-replay bridge ({exc}); reconnects behave as before')

# --- Local PNG staging + bounded /view cache; encoding follows per-image speed policy ---
# Paste before optional timing hooks. Only stock SaveImage + exact app prefix.
# Drive copies finish before the normal result descriptor is returned. A UUID
# subfolder avoids both a growing Drive counter scan and counter reuse after
# local eviction/restart. Nothing here deletes or modifies existing Drive files.
import copy
from contextlib import nullcontext
import nodes

OUTPUT_CACHE_ENABLED = os.environ.get('PHONE_STUDIO_LOCAL_OUTPUT_CACHE', '1') == '1'
_FAST_PNG_COMPRESSION = os.environ.get('PHONE_STUDIO_FAST_PNG_COMPRESSION', '0') == '1'
_OUTPUT_ROOT = Path('/content/phone_studio_output_cache')
_OUTPUT_NAMESPACE = 'phone_studio_v1'
_OUTPUT_MAX_BYTES = int(float(os.environ.get('PHONE_STUDIO_OUTPUT_CACHE_MAX_MB', '256')) * 1024**2)
_OUTPUT_RESERVE_BYTES = int(float(os.environ.get('PHONE_STUDIO_OUTPUT_CACHE_RESERVE_GB', '8')) * 1024**3)
if _OUTPUT_MAX_BYTES <= 0 or _OUTPUT_RESERVE_BYTES < 0:
    raise ValueError('Output cache capacity must be positive and disk reserve cannot be negative')
_OUTPUT_READ_LIMIT = 32 * 1024**2
_OUTPUT_MAX_FILES = 256
_OUTPUT_LOCK = threading.Lock()
_OUTPUT_SAVE_LOCK = threading.Lock()
_OUTPUT_FILES = {}
_OUTPUT_UUID = re.compile(r'^[0-9a-f]{32}$')
_OUTPUT_PNG = re.compile(r'^PhoneStudio_[0-9]+_\.png$')


def _output_stage(name):
    timer = globals().get('_ps_stage')
    return timer(name, gpu=False) if timer is not None else nullcontext()


def _discard_local_output(path):
    # Called only for files enumerated/created under our fixed local namespace.
    try:
        path.unlink(missing_ok=True)
        for directory in (path.parent, path.parent.parent):
            try:
                directory.rmdir()  # Empty only; never recurse or follow links.
            except OSError:
                pass
    except OSError:
        return False
    for key, cached in list(_OUTPUT_FILES.items()):
        if cached == path:
            _OUTPUT_FILES.pop(key, None)
    return True


def _trim_output_cache(incoming=0):
    # Also reclaims this hook's old files after a ComfyUI restart. Never scan
    # Drive or unrelated local files. Called under _OUTPUT_LOCK.
    owned = _OUTPUT_ROOT / _OUTPUT_NAMESPACE
    files = []
    for session in owned.iterdir():
        if session.is_symlink() or not session.is_dir() or not _OUTPUT_UUID.fullmatch(session.name):
            continue
        for batch in session.iterdir():
            if batch.is_symlink() or not batch.is_dir() or not _OUTPUT_UUID.fullmatch(batch.name):
                continue
            for path in batch.iterdir():
                if not path.is_symlink() and path.is_file() and _OUTPUT_PNG.fullmatch(path.name):
                    stat = path.stat()
                    files.append((stat.st_mtime_ns, stat.st_size, path))
    used = sum(size for _, size, _ in files)
    free = shutil.disk_usage(_OUTPUT_ROOT).free
    count = len(files)
    for _, size, path in sorted(files):
        if (used + incoming <= _OUTPUT_MAX_BYTES and count < _OUTPUT_MAX_FILES
                and free - incoming >= _OUTPUT_RESERVE_BYTES):
            break
        if _discard_local_output(path):
            used -= size
            free += size
            count -= 1
    return (used + incoming <= _OUTPUT_MAX_BYTES and count < _OUTPUT_MAX_FILES
            and free - incoming >= _OUTPUT_RESERVE_BYTES)


_drive_save_images = nodes.SaveImage.save_images


@functools.wraps(_drive_save_images)
def _save_images_local(self, images, filename_prefix='ComfyUI', prompt=None, extra_pnginfo=None):
    if (type(self) is not nodes.SaveImage
            or self.type != 'output' or filename_prefix != 'PhoneStudio'
            or self.prefix_append):
        return _drive_save_images(self, images, filename_prefix, prompt, extra_pnginfo)
    # PNG remains lossless. This explicit Fast option changes compressed bytes,
    # not pixel conversion/metadata handling; larger files can cost transfer time.
    # Clone only this call's stock saver, preserving its cached node instance and
    # any nonstandard compression choice. PreviewImage and other prefixes skip it.
    saver = self
    if (_FAST_PNG_COMPRESSION and _phone_studio_speed_extras_enabled()
            and type(getattr(self, 'compress_level', None)) is int
            and self.compress_level == 4
            and (extra_pnginfo is None or isinstance(extra_pnginfo, dict))):
        saver = copy.copy(self)
        saver.compress_level = 1
        extra_pnginfo = dict(extra_pnginfo or {})
        extra_pnginfo['phone_studio_png_encoding'] = {'compress_level': 1, 'lossless': True}
    if not OUTPUT_CACHE_ENABLED:
        return _drive_save_images(saver, images, filename_prefix, prompt, extra_pnginfo)
    # Serialize saves, but release the cache/read lock during encoding and Drive
    # copying so an earlier image can still be displayed during the next save.
    with _OUTPUT_SAVE_LOCK:
        relative = Path(_OUTPUT_NAMESPACE) / INSTANCE_ID / uuid.uuid4().hex
        staging = _OUTPUT_ROOT / relative
        saved_paths = []
        completed = False
        try:
            try:
                with _OUTPUT_LOCK:
                    owned = _OUTPUT_ROOT / _OUTPUT_NAMESPACE
                    _OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
                    if _OUTPUT_ROOT.resolve() != _OUTPUT_ROOT or owned.is_symlink():
                        raise OSError('Local output cache must not be a symbolic link')
                    owned.mkdir(exist_ok=True)
                    # Allow generous PNG/metadata overhead; actual cache bytes
                    # are checked again after the synchronous durable copy.
                    estimate = int(images.numel()) * 2 + 1024**2
                    if not _trim_output_cache(estimate):
                        raise OSError('Local output cache full or below configured disk reserve')
                    staging.mkdir(parents=True, exist_ok=False)
                local_saver = copy.copy(saver)
                local_saver.output_dir = str(_OUTPUT_ROOT)
                with _output_stage('save_png_local'):
                    result = _drive_save_images(
                        local_saver, images, (relative / filename_prefix).as_posix(), prompt, extra_pnginfo)
            except OSError as exc:
                print(f'[phone-studio] Local output staging unavailable ({exc}); saving directly to Drive.')
                return _drive_save_images(saver, images, filename_prefix, prompt, extra_pnginfo)

            descriptors = result.get('ui', {}).get('images', [])
            if not descriptors:
                raise RuntimeError('SaveImage returned no saved image descriptors')
            with _output_stage('save_png_drive_copy'):
                for item in descriptors:
                    if (item.get('type') != 'output' or item.get('subfolder') != relative.as_posix()
                            or not _OUTPUT_PNG.fullmatch(item.get('filename', ''))):
                        raise RuntimeError('SaveImage returned an unsupported output descriptor')
                    source = staging / item['filename']
                    destination = Path(self.output_dir) / relative / item['filename']
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    part = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.part')
                    try:
                        shutil.copyfile(source, part)  # Same PNG bytes, no second encoding.
                        if part.stat().st_size != source.stat().st_size:
                            raise OSError('Incomplete Drive image copy')
                        part.replace(destination)
                    finally:
                        try:
                            part.unlink(missing_ok=True)
                        except OSError:
                            pass
                    saved_paths.append(((item['filename'], item['subfolder'], 'output'), source))
            timing_log = globals().get('_ps_log')
            if callable(timing_log):
                try:
                    timing_log('png_output', 0.0, image_count=len(saved_paths),
                               encoded_bytes=sum(path.stat().st_size for _, path in saved_paths),
                               compress_level=getattr(saver, 'compress_level', None))
                except OSError:
                    pass  # Diagnostic-only; the durable result is already saved.
            # Copy failures above intentionally fail the node. Publish only once
            # every result has its Drive file; cache failure is non-fatal now.
            with _OUTPUT_LOCK:
                for key, path in saved_paths:
                    _OUTPUT_FILES[key] = path
                try:
                    _trim_output_cache()
                except OSError as exc:
                    print(f'[phone-studio] Output cache maintenance skipped ({exc}).')
            completed = True
            return result
        finally:
            if not completed:
                with _OUTPUT_LOCK:
                    # Cleanup only this call's own local PNGs, including a partial
                    # local save. Successfully copied Drive files remain intact.
                    try:
                        if staging.is_dir() and not staging.is_symlink():
                            for path in list(staging.iterdir()):
                                if not path.is_symlink() and _OUTPUT_PNG.fullmatch(path.name):
                                    _discard_local_output(path)
                            try:
                                staging.rmdir()
                            except OSError:
                                pass
                    except OSError:
                        pass


def _read_cached_output(key):
    # Copy to CPU bytes while holding the eviction lock. Returning a FileResponse
    # would race eviction after the handler returns, before aiohttp opens it.
    with _OUTPUT_LOCK:
        path = _OUTPUT_FILES.get(key)
        if path is None:
            return None
        try:
            if path.is_symlink() or path.stat().st_size > _OUTPUT_READ_LIMIT:
                return None
            with path.open('rb') as handle:
                body = handle.read(_OUTPUT_READ_LIMIT + 1)
            return body if len(body) <= _OUTPUT_READ_LIMIT else None
        except OSError:
            _OUTPUT_FILES.pop(key, None)
            return None


@web.middleware
async def _phone_studio_cached_view(request, handler):
    query = request.query
    if (OUTPUT_CACHE_ENABLED and request.method == 'GET' and request.path == '/view'
            and set(query) == {'filename', 'subfolder', 'type'} and len(query) == 3
            and query.get('type') == 'output'
            and not any(name.lower() == 'range' or name.lower().startswith('if-') for name in request.headers)):
        key = (query['filename'], query['subfolder'], 'output')
        try:
            with _output_stage('view_local_read'):
                body = await asyncio.to_thread(_read_cached_output, key)
        except Exception:
            body = None  # A cache-only failure must not hide the Drive image.
        if body is not None:
            return web.Response(body=body, content_type='image/png', headers={
                'Content-Disposition': f'filename="{key[0]}"', 'X-Content-Type-Options': 'nosniff',
            })
    # Includes old images, cache misses, previews/channel conversions, HEAD,
    # conditional/range requests and unsupported query extensions.
    return await handler(request)


if OUTPUT_CACHE_ENABLED:
    PromptServer.instance.app.middlewares.append(_phone_studio_cached_view)
    print(f'[phone-studio] Local PNG staging + /view cache enabled '
          f'({_OUTPUT_MAX_BYTES / 1024**2:g} MiB, {_OUTPUT_RESERVE_BYTES / 1024**3:g} GiB disk reserve).')
if OUTPUT_CACHE_ENABLED or _FAST_PNG_COMPRESSION:
    nodes.SaveImage.save_images = _save_images_local
if _FAST_PNG_COMPRESSION:
    print('[phone-studio] Optional Fast PNG compression requested: level 1 for stock '
          'PhoneStudio saves; pixels stay lossless, file/transfer size may increase.')
# --- End local output cache ----------------------------------------------


# --- Optional timing diagnostics; remove this block to remove the hooks ---
# GPU synchronization changes overlap, so keep this OFF outside diagnosis.
# No prompts, model names, credentials, image data, or response bodies are logged.
if os.environ.get('PHONE_STUDIO_TIMING', '0') == '1':
    import inspect
    import json
    from contextlib import contextmanager
    from contextvars import ContextVar
    import execution
    import nodes

    _PS_CONTEXT = ContextVar('phone_studio_timing', default=None)
    _PS_QUEUED = OrderedDict()
    _PS_TIMING_LOCK = threading.Lock()

    def _ps_log(stage, seconds, context=None, **fields):
        try:
            record = dict(context or _PS_CONTEXT.get() or {})
            record.update(stage=stage, ms=round(seconds * 1000, 1), **fields)
            print('[phone-studio-timing] ' + json.dumps(record, sort_keys=True), flush=True)
        except Exception:
            pass  # Diagnostics must not turn a successful generation into an error.

    def _ps_sync_gpu():
        try:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
                return True
        except Exception:
            pass  # Preserve the original operation's error, including CUDA errors.
        return False

    def _ps_gpu_memory():
        # Boundary snapshots, not per-stage peaks or proof of offload.
        # Never reset allocator peaks: other diagnostics may depend on them.
        try:
            free, total = torch.cuda.mem_get_info()
            return {
                'allocated_mb': round(torch.cuda.memory_allocated() / 1024**2, 1),
                'reserved_mb': round(torch.cuda.memory_reserved() / 1024**2, 1),
                'allocator_peak_mb': round(torch.cuda.max_memory_allocated() / 1024**2, 1),
                'device_free_mb': round(free / 1024**2, 1),
                'device_total_mb': round(total / 1024**2, 1),
            }
        except Exception:
            return {}  # Diagnostics never hide the original error.

    @contextmanager
    def _ps_stage(stage, gpu=True):
        synced_before = _ps_sync_gpu() if gpu else False
        memory_before = _ps_gpu_memory() if gpu else {}
        started = time.perf_counter()
        completed = False
        try:
            yield
            completed = True
        finally:
            synced_after = _ps_sync_gpu() if gpu else False
            seconds = time.perf_counter() - started
            memory_after = _ps_gpu_memory() if gpu else {}
            _ps_log(stage, seconds,
                    status='returned' if completed else 'raised',
                    gpu_synced=synced_before and synced_after,
                    gpu_memory_before=memory_before, gpu_memory_after=memory_after)

    def _ps_wrap_node(original, stage):
        def stage_for(instance):
            # PreviewImage inherits SaveImage's method, but writes to /content.
            if stage == 'save_png_drive' and getattr(instance, 'type', '') == 'temp':
                return 'save_png_preview'
            return stage

        if inspect.iscoroutinefunction(original):
            @functools.wraps(original)
            async def wrapped(self, *args, **kwargs):
                if _PS_CONTEXT.get() is None:
                    return await original(self, *args, **kwargs)
                with _ps_stage(stage_for(self)):
                    return await original(self, *args, **kwargs)
        else:
            @functools.wraps(original)
            def wrapped(self, *args, **kwargs):
                if _PS_CONTEXT.get() is None:
                    return original(self, *args, **kwargs)
                with _ps_stage(stage_for(self)):
                    return original(self, *args, **kwargs)
        wrapped._phone_studio_timing = True
        return wrapped

    # Stage times include ComfyUI's loading/offloading inside each node. In
    # particular checkpoint_load includes the local-cache lookup/copy below.
    # SaveImage combines conversion, PNG encoding, directory work and Drive I/O.
    for _class_name, _stage in {
        'CheckpointLoaderSimple': 'checkpoint_load',
        'LoraLoader': 'lora_load',
        'CLIPTextEncode': 'text_encode',
        'KSampler': 'sampling',
        'VAEDecode': 'vae_decode',
        'SaveImage': 'save_png_drive',
    }.items():
        _class = nodes.NODE_CLASS_MAPPINGS.get(_class_name)
        _method = getattr(_class, 'FUNCTION', '')
        _original = getattr(_class, _method, None)
        if inspect.isfunction(_original) and not getattr(_original, '_phone_studio_timing', False):
            setattr(_class, _method, _ps_wrap_node(_original, _stage))
        elif not getattr(_original, '_phone_studio_timing', False):
            print(f'[phone-studio] Timing hook unavailable: {_class_name}')

    # Nested within checkpoint/LoRA loading; do not add this to the stage total.
    if not getattr(local_model_path, '_phone_studio_timing', False):
        _ps_original_model_path = local_model_path

        @functools.wraps(_ps_original_model_path)
        def local_model_path(filename):
            if _PS_CONTEXT.get() is None:
                return _ps_original_model_path(filename)
            with _ps_stage('model_cache_lookup_copy', gpu=False):
                return _ps_original_model_path(filename)

        local_model_path._phone_studio_timing = True

    # Capture before insertion, because the worker can start before /prompt
    # returns. Bound the map so cancelled/removed prompts cannot leak memory.
    if not getattr(execution.PromptQueue.put, '_phone_studio_timing', False):
        _ps_original_put = execution.PromptQueue.put
        @functools.wraps(_ps_original_put)
        def _ps_put(self, item, *args, **kwargs):
            prompt_id = item[1] if isinstance(item, (tuple, list)) and len(item) > 1 else None
            if prompt_id is not None:
                with _PS_TIMING_LOCK:
                    _PS_QUEUED[prompt_id] = time.perf_counter()
                    while len(_PS_QUEUED) > 128:
                        _PS_QUEUED.popitem(last=False)
            try:
                return _ps_original_put(self, item, *args, **kwargs)
            except BaseException:
                with _PS_TIMING_LOCK:
                    _PS_QUEUED.pop(prompt_id, None)
                raise

        _ps_put._phone_studio_timing = True
        execution.PromptQueue.put = _ps_put

    @contextmanager
    def _ps_execution_scope(executor, bound):
        prompt_id = bound.arguments.get('prompt_id')
        extra = bound.arguments.get('extra_data') or {}
        if not isinstance(extra, dict):
            extra = {}
        context = {
            'prompt_id': str(prompt_id or ''),
            'request_id': str(extra.get('phone_studio_request_id') or ''),
            'warmup': bool(extra.get('phone_studio_warmup')),
        }
        token = _PS_CONTEXT.set(context)
        with _PS_TIMING_LOCK:
            queued_at = _PS_QUEUED.pop(prompt_id, None)
        if queued_at is not None:
            _ps_log('queue_wait', time.perf_counter() - queued_at)
        try:
            with _ps_stage('execute_total'):
                yield
        finally:
            if queued_at is not None:
                _ps_log('queued_to_execute_end', time.perf_counter() - queued_at,
                        success=bool(getattr(executor, 'success', False)))
            _PS_CONTEXT.reset(token)

    # Current ComfyUI's sync execute() calls execute_async(). Older versions
    # expose only execute(); preserve the callable's synchronous/async shape.
    def _ps_wrap_executor(original):
        signature = inspect.signature(original)
        if not {'prompt_id', 'extra_data'}.issubset(signature.parameters):
            print('[phone-studio] Executor timing unavailable: unsupported signature')
            return original
        if inspect.iscoroutinefunction(original):
            @functools.wraps(original)
            async def wrapped(self, *args, **kwargs):
                with _ps_execution_scope(self, signature.bind(self, *args, **kwargs)):
                    return await original(self, *args, **kwargs)
        else:
            @functools.wraps(original)
            def wrapped(self, *args, **kwargs):
                with _ps_execution_scope(self, signature.bind(self, *args, **kwargs)):
                    return original(self, *args, **kwargs)
        wrapped._phone_studio_timing = True
        return wrapped

    _execute_name = 'execute_async' if hasattr(execution.PromptExecutor, 'execute_async') else 'execute'
    _execute = getattr(execution.PromptExecutor, _execute_name)
    if not getattr(_execute, '_phone_studio_timing', False):
        setattr(execution.PromptExecutor, _execute_name, _ps_wrap_executor(_execute))

    # These are handler times only. /view returns FileResponse before its file
    # is transferred; phone-side image-ready timing covers transfer and decode.
    @web.middleware
    async def _ps_http_timing(request, handler):
        stages = {'/phone-studio/models': 'http_models', '/prompt': 'http_prompt',
                  '/view': 'http_view_handler_only'}
        stage = stages.get(request.path)
        if stage is None:
            return await handler(request)
        started = time.perf_counter()
        response = None
        try:
            response = await handler(request)
            return response
        finally:
            _ps_log(stage, time.perf_counter() - started,
                    status=getattr(response, 'status', 'raised'))

    if not any(getattr(item, '__name__', '') == '_ps_http_timing'
               for item in PromptServer.instance.app.middlewares):
        PromptServer.instance.app.middlewares.append(_ps_http_timing)
    print('[phone-studio] Timing ON; GPU sync adds overhead. Missing node stages mean cached/not run, not zero ms.')
# --- End optional timing diagnostics --------------------------------------

NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}
'''
# --- Local bridge publication ---
CUSTOM_NODE_INVENTORY = _worker_setup['local_custom_nodes'](CUSTOM_NODES_ROOT,
    RUNTIME_CUSTOM_NODES_ROOT, known_bridge_hashes=('d38fdc9a78cb814c8be77510fac30ea4c98ca3baaab6aa55bc123480a5d6d9e9', '243165926d9eb0e513484a072e3226f9e0a3ce548ff0bac95c3b4e569b824fab'))
_worker_setup['atomic_text'](BRIDGE_DIR / '__init__.py', BRIDGE_CODE)
WORKER_BUILD_FINGERPRINT = '69956fedcc83904e0b1d73a5fce0ddefa0a8dd553bc237fd4f33985e9a8d325c'
print('Persistent user custom nodes retained:', CUSTOM_NODE_INVENTORY)
print('Instance-local Phone Studio bridge:', WORKER_BUILD_FINGERPRINT)


In [ ]:
# Start ComfyUI and create a password-protected ngrok endpoint.
import socket
import time
import requests

from pyngrok import ngrok

def publish_completed_warmup(session, base_url, prompt_id, expected_instance_id):
    # Called only after confirmed completion and decoded preview output.
    # Optional file IPC avoids another bridge write API. No token is stored.
    import json
    temporary = COMPLETED_WARMUP_FILE.with_name(COMPLETED_WARMUP_FILE.name + '.' + uuid.uuid4().hex + '.part')
    try:
        response = session.get(base_url + '/phone-studio/health', timeout=8)
        response.raise_for_status()
        health = response.json()
        instance_id = health.get('instanceId', '')
        if (not health.get('ok') or not isinstance(instance_id, str) or not instance_id
                or not expected_instance_id or instance_id != expected_instance_id):
            raise ValueError('Warm-up runtime changed or was not confirmed')
        record = {'instanceId': instance_id, 'completedWarmup': {
            'model': DEFAULT_CHECKPOINT, 'width': WARMUP_WIDTH, 'height': WARMUP_HEIGHT,
            'sampler': WARMUP_SAMPLER, 'scheduler': WARMUP_SCHEDULER,
            'guidanceMode': 'cfg', 'promptId': prompt_id,
        }}
        temporary.write_text(json.dumps(record), encoding='utf-8')
        temporary.replace(COMPLETED_WARMUP_FILE)
        print('Startup warm-up confirmed; matching phone connection checks can reuse it.')
    except Exception:
        print('Startup warm-up reuse metadata unavailable; the phone may run its own preview.')
    finally:
        try:
            temporary.unlink(missing_ok=True)
        except OSError:
            pass


def warmup_default_checkpoint(base_url):
    # PreviewImage is an output node: this runs VAE decoding too, while writing
    # only a disposable local preview, never a warm-up image to the Drive gallery.
    prompt = {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': DEFAULT_CHECKPOINT}},
        '2': {'class_type': 'CLIPTextEncode', 'inputs': {'text': WARMUP_PROMPT, 'clip': ['1', 1]}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['1', 1]}},
        '4': {'class_type': 'EmptyLatentImage', 'inputs': {
            'width': WARMUP_WIDTH, 'height': WARMUP_HEIGHT, 'batch_size': 1,
        }},
        '5': {'class_type': 'KSampler', 'inputs': {
            'model': ['1', 0], 'seed': 170, 'steps': WARMUP_STEPS, 'cfg': 6.5,
            'sampler_name': WARMUP_SAMPLER, 'scheduler': WARMUP_SCHEDULER,
            'positive': ['2', 0], 'negative': ['3', 0], 'latent_image': ['4', 0], 'denoise': 1.0,
        }},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5', 0], 'vae': ['1', 2]}},
        '7': {'class_type': 'PreviewImage', 'inputs': {'images': ['6', 0]}},
    }
    started = time.monotonic()
    print(f'Warming up {DEFAULT_CHECKPOINT}: {WARMUP_WIDTH} x {WARMUP_HEIGHT}, {WARMUP_STEPS} steps...')
    with requests.Session() as session:
        session.headers.update({'X-Phone-Studio-Token': PHONE_STUDIO_TOKEN})
        startup_instance_id = None
        try:
            health_response = session.get(base_url + '/phone-studio/health', timeout=8)
            health_response.raise_for_status()
            health = health_response.json()
            if health.get('ok') and isinstance(health.get('instanceId'), str):
                startup_instance_id = health['instanceId']
        except Exception:
            pass  # Reuse proof is optional; the normal warm-up still runs.
        response = session.post(base_url + '/prompt', json={
            'prompt': prompt,
            'client_id': 'phone-studio-startup-warmup',
            'extra_data': {'phone_studio_warmup': True},
        }, timeout=30)
        if response.status_code != 200:
            raise RuntimeError(f'Warm-up prompt was rejected: {response.text[:1500]}')
        prompt_id = response.json().get('prompt_id')
        if not prompt_id:
            raise RuntimeError(f'Warm-up did not receive a prompt ID: {response.text[:1500]}')
        next_update = started + 15
        while time.monotonic() - started < WARMUP_TIMEOUT_SECONDS:
            if comfy_process.poll() is not None:
                raise RuntimeError('ComfyUI exited during model warm-up. Inspect the output above.')
            response = session.get(base_url + f'/history/{prompt_id}', timeout=30)
            response.raise_for_status()
            entry = response.json().get(prompt_id)
            if entry is not None:
                status = entry.get('status') or {}
                if status.get('status_str') == 'error':
                    raise RuntimeError(f'Model warm-up failed: {str(status.get("messages", []))[:2000]}')
                if status.get('completed'):
                    if not any(output.get('images') for output in (entry.get('outputs') or {}).values()):
                        raise RuntimeError('Warm-up completed without a decoded preview; inspect the ComfyUI log.')
                    print(f'Warm-up complete in {time.monotonic() - started:.1f}s. Default model is ready.')
                    publish_completed_warmup(session, base_url, prompt_id, startup_instance_id)
                    return
                raise RuntimeError(f'Warm-up stopped before completion: {str(status)[:1500]}')
            if time.monotonic() >= next_update:
                print(f'Warm-up still running ({time.monotonic() - started:.0f}s); first-time model loading can take a while.')
                next_update = time.monotonic() + 15
            time.sleep(2)
    raise TimeoutError('Model warm-up timed out. Inspect the log or increase WARMUP_TIMEOUT_SECONDS.')


PORT = 8188
WORKER_SERVER_INSTANCE_ID = uuid.uuid4().hex
env = os.environ.copy()
env.update({
    'PHONE_STUDIO_FOOOCUS_READY': '1' if FOOOCUS_SETUP.get('ready') else '0',
    'PHONE_STUDIO_FOOOCUS_SOURCE_REVISION': FOOOCUS_SETUP.get('sourceRevision', ''),
    'PHONE_STUDIO_COMPATIBILITY_MODE': '1' if COMPATIBILITY_MODE else '0',
    'PHONE_STUDIO_RUNTIME_ID': WORKER_RUNTIME_ID,
    'PHONE_STUDIO_WORKER_BUILD': WORKER_BUILD_FINGERPRINT,
    'PHONE_STUDIO_COMFY_REVISION': COMFY_REVISION,
    'PHONE_STUDIO_EFFECTIVE_OPTIONS': json.dumps(WORKER_OPTIONS['effective']),
    'PHONE_STUDIO_ATTENTION': 'sage' if SAGE_ATTENTION_AVAILABLE else 'pytorch',
    'PHONE_STUDIO_EXPECTED_INSTANCE_ID': WORKER_SERVER_INSTANCE_ID,
    'PHONE_STUDIO_MODELS_ROOT': str(MODELS_ROOT),
    'PHONE_STUDIO_CATALOG_CACHE_SECONDS': str(MODEL_CATALOG_CACHE_SECONDS),
    'PHONE_STUDIO_TIMING': '1' if PHONE_STUDIO_TIMING else '0',
    'PHONE_STUDIO_LOCAL_OUTPUT_CACHE': '1' if LOCAL_OUTPUT_CACHE else '0',
    'PHONE_STUDIO_FAST_PNG_COMPRESSION': '1' if FAST_PNG_COMPRESSION else '0',
    'PHONE_STUDIO_OUTPUT_CACHE_MAX_MB': str(OUTPUT_CACHE_MAX_MB),
    'PHONE_STUDIO_OUTPUT_CACHE_RESERVE_GB': str(MODEL_CACHE_RESERVE_GB),
    'PHONE_STUDIO_FAST_DENOISE_CACHE': '1' if FAST_DENOISE_CACHE else '0',
    'PHONE_STUDIO_FAST_CONDITIONING_CACHE': '1' if FAST_CONDITIONING_CACHE else '0',
    'PHONE_STUDIO_T4_FLASH_ATTENTION': '1' if FAST_T4_FLASH_ATTENTION and T4_FLASH_ATTENTION_AVAILABLE else '0',
    'PHONE_STUDIO_DEFAULT_CHECKPOINT': DEFAULT_CHECKPOINT,
    'PHONE_STUDIO_COMPLETED_WARMUP_FILE': str(COMPLETED_WARMUP_FILE),
    'PHONE_STUDIO_TOKEN': PHONE_STUDIO_TOKEN,
    'CIVITAI_API_KEY': CIVITAI_API_KEY,
    'PHONE_STUDIO_AUTO_COMPILE': PHONE_STUDIO_AUTO_COMPILE,
    'PHONE_STUDIO_LOCAL_MODEL_CACHE': '1' if LOCAL_MODEL_CACHE else '0',
    'PHONE_STUDIO_MODEL_CACHE_ROOT': str(MODEL_CACHE_ROOT),
    'PHONE_STUDIO_MODEL_CACHE_MAX_GB': str(MODEL_CACHE_MAX_GB),
    'PHONE_STUDIO_MODEL_CACHE_RESERVE_GB': str(MODEL_CACHE_RESERVE_GB),
})

attention_flag = ['--use-sage-attention'] if SAGE_ATTENTION_AVAILABLE else ['--use-pytorch-cross-attention']
print('Attention default/fallback:', attention_flag[0])
if FAST_T4_FLASH_ATTENTION and T4_FLASH_ATTENTION_AVAILABLE:
    print('Optional T4 Fast attention requested for eligible SDXL calls; see bridge startup/use logs.')

# Stop an earlier server from this notebook before allocating another model.
previous_process = globals().get('comfy_process')
if previous_process is not None and previous_process.poll() is None:
    previous_process.terminate()
    try:
        previous_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        previous_process.kill()
        previous_process.wait(timeout=5)

# Refuse an unrelated local port owner without terminating it. A closed probe
# socket does not reserve the port; authenticated instance validation below
# also catches a process that wins a later bind race.
try:
    with socket.create_connection(('127.0.0.1', PORT), timeout=1):
        raise RuntimeError('Worker port is already occupied by an unowned process; it was left running.')
except OSError:
    pass

# Use named features: bare --fast would also enable unrelated precision changes.
speed_flags = ['--fast', 'autotune']
cli_source = (RUNTIME_ROOT / 'comfy' / 'cli_args.py').read_text(encoding='utf-8')
if COMPATIBILITY_MODE:
    if '--disable-comfy-compiler' not in cli_source:
        raise RuntimeError('Pinned ComfyUI compiler-disable interface is missing; preserve the checkout.')
    speed_flags.append('--disable-comfy-compiler')
if FAST_FP16_ACCUMULATION:
    if 'fp16_accumulation' not in cli_source or not hasattr(torch.backends.cuda.matmul, 'allow_fp16_accumulation'):
        print('Fast FP16 accumulation unavailable on this ComfyUI/PyTorch; stock accumulation retained.')
    else:
        speed_flags.append('fp16_accumulation')
        print('EXPERIMENTAL Fast FP16 accumulation enabled: check images for changes/non-finite output.')
# Request configured precision experiments individually when CLI support exists.
for enabled, flag, label in (
    (FAST_FP16_INTERMEDIATES, '--fp16-intermediates', 'FP16 intermediate buffers (noise/RGB rounding changes)'),
    (FAST_FP16_VAE, '--fp16-vae', 'FP16 VAE decode (black/non-finite output is possible)'),
):
    if enabled:
        if flag not in cli_source:
            print('Fast precision unavailable; stock precision retained:', label)
        else:
            speed_flags.append(flag)
            print('EXPERIMENTAL Fast precision enabled:', label)
if CHANNELS_LAST:
    speed_flags.append('--force-channels-last')
if VRAM_MODE == 'high':
    speed_flags.append('--highvram')
elif VRAM_MODE == 'legacy':
    if '--disable-dynamic-vram' not in cli_source:
        raise RuntimeError('This ComfyUI no longer supports legacy VRAM mode; set VRAM_MODE to auto.')
    speed_flags.append('--disable-dynamic-vram')
# In auto mode, leave ComfyUI's smart memory/dynamic VRAM defaults enabled.
# OkHttp negotiates/decodes gzip; stock middleware excludes PNG/binary output.
if COMPRESS_JSON_RESPONSES:
    if '--enable-compress-response-body' in cli_source:
        speed_flags.append('--enable-compress-response-body')
        print('JSON/text response compression enabled; PNGs are unchanged.')
    else:
        print('JSON compression flag unavailable; keeping uncompressed responses.')
print('Speed settings: FP16 diffusion/text, cuDNN autotune, VRAM mode =', VRAM_MODE)
comfy_process = subprocess.Popen([
    sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
    '--fp16-unet', '--fp16-text-enc', '--enable-cors-header',
    *attention_flag,
    *speed_flags,
], cwd=RUNTIME_ROOT, env=env)

# --- Private worker connection output ---
def show_worker_pairing():
    global WORKER_PAIRING_CODE
    connection = {'version': 1, 'kind': 'worker', 'name': WORKER_NAME,
        'workerId': WORKER_ID, 'serverUrl': SERVER_URL, 'tunnelUser': TUNNEL_USER,
        'tunnelPassword': TUNNEL_PASSWORD, 'studioToken': PHONE_STUDIO_TOKEN}
    WORKER_PAIRING_CODE = _pairing['encode_pairing'](connection)
    _pairing['display_pairing'](connection, with_qr=False)
# --- End private worker connection output ---

SERVER_URL = None
try:
    deadline = time.time() + 180
    while time.time() < deadline:
        if comfy_process.poll() is not None:
            raise RuntimeError('ComfyUI exited before it opened. Rerun the previous cells and inspect the Colab output.')
        try:
            local_health = requests.get(f'http://127.0.0.1:{PORT}/phone-studio/health',
                headers={'X-Phone-Studio-Token': PHONE_STUDIO_TOKEN}, timeout=2)
            if local_health.status_code in (401, 403):
                raise RuntimeError('Local worker authentication conflict; unrelated process preserved.')
            local_health.raise_for_status()
            _worker_setup['confirm_runtime_health'](local_health.json(), WORKER_SERVER_INSTANCE_ID, WORKER_RUNTIME_ID)
            break
        except (OSError, requests.exceptions.RequestException):
            time.sleep(2)
    else:
        raise TimeoutError('ComfyUI did not open its port within three minutes.')

    if WARMUP_ON_START:
        warmup_default_checkpoint(f'http://127.0.0.1:{PORT}')

    # Optional: pin the ngrok URL so it never changes across tunnel/runtime restarts.
    # Every ngrok account gets one free dev domain automatically - find yours at
    # ngrok dashboard -> Gateway -> Domains, then add it as the NGROK_DOMAIN Colab secret
    # (the provider assigns your dev domain). The current free plan permits
    # three concurrent agents/endpoints and one dev domain per account; it
    # does not provide four distinct free runtime endpoints or random URLs.
    # Workers need individually reachable permitted network addresses.
    NGROK_DOMAIN = colab_secret('NGROK_DOMAIN')
    tunnel_kwargs = {'auth': f'{TUNNEL_USER}:{TUNNEL_PASSWORD}'}
    if NGROK_DOMAIN:
        tunnel_kwargs['domain'] = NGROK_DOMAIN
        print('Using reserved ngrok domain:', NGROK_DOMAIN)
    else:
        print('No NGROK_DOMAIN secret set - using the domain permitted by this ngrok account.')
        print('Check the assigned domain and endpoint/agent allowance in your ngrok dashboard; do not reuse one worker address for three workers.')

    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    previous_tunnel = globals().get('tunnel')
    if previous_tunnel is not None:
        ngrok.disconnect(previous_tunnel.public_url)
    tunnel = ngrok.connect(PORT, 'http', **tunnel_kwargs)
    SERVER_URL = tunnel.public_url

    show_worker_pairing()
    print('Keep this Colab tab running while you generate images.')
    print('This cell will now stay busy on purpose instead of finishing.')
    print('This loop monitors ComfyUI and reconnects a dropped tunnel.')
    print('It cannot prevent Colab timeouts or runtime reclamation; stop it when finished.')
    print('Stop this cell (Runtime > Interrupt) to shut everything down; rerun it to relaunch.')

    heartbeat = 0
    while True:
        time.sleep(60)
        heartbeat += 1

        if comfy_process.poll() is not None:
            raise RuntimeError('ComfyUI process exited. Rerun the previous cells, then this one, to relaunch.')

        active_urls = [t.public_url for t in ngrok.get_tunnels()]
        if SERVER_URL not in active_urls:
            print('ngrok tunnel dropped, reconnecting...')
            tunnel = ngrok.connect(PORT, 'http', **tunnel_kwargs)
            SERVER_URL = tunnel.public_url
            print('Worker tunnel restored. Replace only this worker’s private code in the router, then rerun idle router config/serve.')
            show_worker_pairing()

        if heartbeat % 5 == 0:
            print(f'[{time.strftime("%H:%M:%S")}] still running -> {SERVER_URL}')
except KeyboardInterrupt:
    print('Stopped manually. Run this cell again to relaunch ComfyUI and get a fresh tunnel.')
finally:
    # Release GPU memory on interruption/error so a rerun cannot leave two servers.
    if comfy_process.poll() is None:
        comfy_process.terminate()
        try:
            comfy_process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            comfy_process.kill()
            comfy_process.wait(timeout=5)
    try:
        if SERVER_URL:
            ngrok.disconnect(SERVER_URL)
    except Exception:
        pass
